# Chapter 6 — Compacting Context at Settled Boundaries

This chapter makes long conversations continuable without deleting their durable evidence or confusing context recovery with ordinary retries.

## Goal and Previous Limitation

Chapter 5 can restore every settled message, but every later model request still receives the entire active branch. A long Session eventually crosses its configured context capacity. We begin from the immutable Chapter 5 Checkpoint and add Compaction as an AgentSession operation.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
CHAPTER_5 = ROOT / 'course' / 'checkpoints' / 'ch05'
sys.path.insert(0, str(CHAPTER_5 / 'src'))
import agent_harness as chapter5

assert hasattr(chapter5, 'AgentSession')
assert not hasattr(chapter5, 'CompactionPolicy')
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]
sys.path.remove(str(CHAPTER_5 / 'src'))


## Conceptual Model

`AgentSession.compact(...)` is the application interface. Behind that seam, `CompactionStrategy` finds a structurally valid cut, the Runtime asks the same ModelAdapter for a summary under `ModelOperation.COMPACTION`, and the Session atomically appends a `CompactionCheckpoint`.

The checkpoint contains a versioned `StructuredSummary`, estimated tokens-before, summary usage, trigger, and a materialized Retained Tail. `history()` keeps navigating original messages; `effective_history()` gives the next model request the summary, retained messages, and later settlements.

## Minimal Execution

The following Export Cells introduce the Compaction module and replace only the Chapter 5 modules whose interfaces evolve. Each cell owns one complete file.

In [ ]:
COMPACTION_SOURCE = '"""Session Compaction policy, planning, and durable checkpoint contracts."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nfrom typing import Protocol, TypeAlias\n\nfrom .model import (\n    AgentMessage,\n    ModelContextMessage,\n    ModelSpec,\n    Role,\n    TextContent,\n    ToolCallContent,\n    Usage,\n    to_model_messages,\n)\nfrom .tools import ToolResultMessage\n\n\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\n\n\nclass CompactionTrigger(str, Enum):\n    MANUAL = "manual"\n    THRESHOLD = "threshold"\n    OVERFLOW = "overflow"\n\n\nclass CompactionWarningCode(str, Enum):\n    THRESHOLD_FAILED = "threshold_failed"\n    CONTEXT_WINDOW_UNKNOWN = "context_window_unknown"\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionWarning:\n    code: CompactionWarningCode\n    message: str\n\n\n@dataclass(frozen=True, slots=True)\nclass StructuredSummary:\n    text: str\n    focus: str | None = None\n    schema_version: int = 1\n\n    def __post_init__(self) -> None:\n        if not self.text.strip():\n            raise ValueError("a Compaction summary cannot be empty")\n        if self.focus is not None and not self.focus.strip():\n            raise ValueError("Compaction focus cannot be empty")\n        if self.schema_version != 1:\n            raise ValueError("unsupported Compaction summary schema version")\n\n    def as_message(self) -> AgentMessage:\n        focus = f"Focus: {self.focus}\\n" if self.focus is not None else ""\n        return AgentMessage.text(\n            Role.SYSTEM,\n            "Compaction checkpoint (schema 1)\\n"\n            f"{focus}Summary:\\n{self.text}",\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionCheckpoint:\n    trigger: CompactionTrigger\n    summary: StructuredSummary\n    tokens_before: int\n    summary_usage: Usage | None\n    retained_tail: tuple[ConversationMessage, ...]\n\n    def __post_init__(self) -> None:\n        if self.tokens_before < 0:\n            raise ValueError("tokens_before cannot be negative")\n        if not self.retained_tail:\n            raise ValueError("a Compaction checkpoint requires a Retained Tail")\n        to_model_messages(self.retained_tail)\n\n    def model_context(self) -> tuple[ModelContextMessage, ...]:\n        return to_model_messages((self.summary.as_message(), *self.retained_tail))\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionPlan:\n    source: tuple[ConversationMessage, ...]\n    retained_tail: tuple[ConversationMessage, ...]\n    tokens_before: int\n\n\nclass TokenEstimator(Protocol):\n    def estimate(self, messages: Sequence[ConversationMessage]) -> int: ...\n\n\nclass CharacterTokenEstimator:\n    """Deterministic offline estimate with explicit estimated provenance."""\n\n    def estimate(self, messages: Sequence[ConversationMessage]) -> int:\n        characters = 0\n        for message in messages:\n            if isinstance(message, ToolResultMessage):\n                characters += len(message.result.content) + len(message.tool_name)\n                continue\n            for block in message.content:\n                if isinstance(block, TextContent):\n                    characters += len(block.text)\n                elif isinstance(block, ToolCallContent):\n                    characters += len(block.name) + len(block.arguments)\n            characters += 8\n        return 0 if not messages else max(1, (characters + 3) // 4)\n\n\n@dataclass(frozen=True, slots=True)\nclass ResolvedCompactionPolicy:\n    context_window: int | None\n    reserve_tokens: int | None\n    keep_recent_tokens: int\n\n    @property\n    def threshold_tokens(self) -> int | None:\n        if self.context_window is None or self.reserve_tokens is None:\n            return None\n        return max(0, self.context_window - self.reserve_tokens)\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionPolicy:\n    reserve_tokens: int | None = None\n    keep_recent_tokens: int | None = None\n\n    def __post_init__(self) -> None:\n        for name, value in (\n            ("reserve_tokens", self.reserve_tokens),\n            ("keep_recent_tokens", self.keep_recent_tokens),\n        ):\n            if value is not None and (\n                isinstance(value, bool) or not isinstance(value, int) or value <= 0\n            ):\n                raise ValueError(f"{name} must be a positive integer when supplied")\n\n    def resolve(self, model: ModelSpec) -> ResolvedCompactionPolicy:\n        window = model.context_window\n        reserve = self.reserve_tokens\n        keep = self.keep_recent_tokens\n        if window is not None:\n            if reserve is None:\n                reserve = min(\n                    16_384,\n                    max(model.max_output_tokens, window // 8),\n                )\n            if keep is None:\n                keep = min(20_000, window // 4)\n        return ResolvedCompactionPolicy(\n            window,\n            reserve,\n            keep if keep is not None else 20_000,\n        )\n\n\nclass CompactionStrategy:\n    """Choose a Retained Tail without splitting a Tool Call batch."""\n\n    def plan(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        keep_recent_tokens: int,\n        estimator: TokenEstimator,\n    ) -> CompactionPlan:\n        if (\n            isinstance(keep_recent_tokens, bool)\n            or not isinstance(keep_recent_tokens, int)\n            or keep_recent_tokens < 0\n        ):\n            raise ValueError("keep_recent_tokens must be a non-negative integer")\n        if not callable(getattr(estimator, "estimate", None)):\n            raise TypeError("estimator must implement TokenEstimator.estimate")\n        accepted = tuple(messages)\n        if len(accepted) < 2:\n            raise ValueError("Compaction requires history before the Retained Tail")\n        groups = self._structural_groups(accepted)\n        if len(groups) < 2:\n            raise ValueError("Compaction requires history before the Retained Tail")\n        retained_groups: list[tuple[ConversationMessage, ...]] = []\n        retained_tokens = 0\n        for group in reversed(groups):\n            group_tokens = estimator.estimate(group)\n            if group_tokens < 0:\n                raise ValueError("TokenEstimator cannot return negative values")\n            if retained_groups and retained_tokens + group_tokens > keep_recent_tokens:\n                break\n            retained_groups.append(group)\n            retained_tokens += group_tokens\n            if retained_tokens >= keep_recent_tokens:\n                break\n        tail = tuple(\n            message\n            for group in reversed(retained_groups)\n            for message in group\n        )\n        if len(tail) >= len(accepted):\n            first = groups[0]\n            tail = accepted[len(first) :]\n        if not tail:\n            tail = groups[-1]\n        tokens_before = estimator.estimate(accepted)\n        if tokens_before < 0:\n            raise ValueError("TokenEstimator cannot return negative values")\n        return CompactionPlan(\n            source=accepted[: -len(tail)],\n            retained_tail=tail,\n            tokens_before=tokens_before,\n        )\n\n    @staticmethod\n    def _structural_groups(\n        messages: tuple[ConversationMessage, ...],\n    ) -> tuple[tuple[ConversationMessage, ...], ...]:\n        groups: list[tuple[ConversationMessage, ...]] = []\n        index = 0\n        while index < len(messages):\n            message = messages[index]\n            if isinstance(message, ToolResultMessage):\n                raise ValueError("a settled history cannot contain an orphan ToolResult")\n            calls = (\n                tuple(\n                    block\n                    for block in message.content\n                    if isinstance(block, ToolCallContent)\n                )\n                if isinstance(message, AgentMessage)\n                else ()\n            )\n            if not calls:\n                groups.append((message,))\n                index += 1\n                continue\n            expected = {call.id for call in calls}\n            end = index + 1\n            while end < len(messages):\n                result = messages[end]\n                if not isinstance(result, ToolResultMessage):\n                    break\n                if result.tool_call_id not in expected:\n                    raise ValueError(\n                        "a settled history cannot contain an orphan ToolResult"\n                    )\n                expected.remove(result.tool_call_id)\n                end += 1\n            if expected:\n                raise ValueError("a settled history cannot contain an orphan Tool Call")\n            groups.append(messages[index:end])\n            index = end\n        return tuple(groups)\n'


In [ ]:
MODEL_SOURCE = '"""Provider-neutral message and scripted model contracts for Chapter 3 work."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import AsyncIterator, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom types import MappingProxyType\nfrom typing import Any, Literal, Protocol, TypeAlias, cast\nfrom urllib.parse import urlparse\n\nimport openai\n\n\nclass Role(str, Enum):\n    SYSTEM = "system"\n    USER = "user"\n    ASSISTANT = "assistant"\n    TOOL = "tool"\n\n\nclass StopReason(str, Enum):\n    COMPLETE = "complete"\n    TOOL_USE = "tool_use"\n    LENGTH = "length"\n    CONTENT_FILTER = "content_filter"\n    ERROR = "error"\n    ABORTED = "aborted"\n    OTHER = "other"\n\n\nclass ModelOperation(str, Enum):\n    RUN = "run"\n    COMPACTION = "compaction"\n\n\nclass ModelErrorCode(str, Enum):\n    AUTHENTICATION = "authentication"\n    REQUEST = "request"\n    SCHEMA = "schema"\n    RATE_LIMIT = "rate_limit"\n    TIMEOUT = "timeout"\n    CONNECTION = "connection"\n    SERVER = "server"\n    PROVIDER = "provider"\n    CONTEXT_OVERFLOW = "context_overflow"\n    COMPACTION_FAILED = "compaction_failed"\n\n\nclass UnsupportedContentError(ValueError):\n    """Raised before provider I/O for unsupported content."""\n\n\nclass ModelProtocolError(RuntimeError):\n    """Raised when an adapter violates the provider-neutral stream contract."""\n\n\n@dataclass(frozen=True, slots=True)\nclass TextContent:\n    text: str\n    type: Literal["text"] = field(default="text", init=False)\n    schema_version: Literal[1] = field(default=1, init=False)\n\n    def __post_init__(self) -> None:\n        if not isinstance(self.text, str):\n            raise TypeError("TextContent.text must be a string")\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolCallContent:\n    id: str\n    name: str\n    arguments: str\n    type: Literal["tool_call"] = field(default="tool_call", init=False)\n    schema_version: Literal[1] = field(default=1, init=False)\n\n    def __post_init__(self) -> None:\n        if not self.id or not self.name:\n            raise ValueError("a Tool Call requires non-empty id and name")\n        if not isinstance(self.arguments, str):\n            raise TypeError("ToolCallContent.arguments must be a JSON string")\n\n\nContentBlock: TypeAlias = TextContent | ToolCallContent\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelMessage:\n    role: Role\n    content: tuple[ContentBlock, ...]\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelToolResultMessage:\n    tool_call_id: str\n    tool_name: str\n    content: str\n    is_error: bool = False\n    role: Literal[Role.TOOL] = field(default=Role.TOOL, init=False)\n\n\nModelContextMessage: TypeAlias = ModelMessage | ModelToolResultMessage\n\n\n@dataclass(frozen=True, slots=True)\nclass AgentMessage:\n    role: Role\n    content: tuple[ContentBlock, ...]\n\n    @classmethod\n    def text(cls, role: Role, text: str) -> AgentMessage:\n        return cls(role=role, content=(TextContent(text),))\n\n    def to_model(self) -> ModelMessage:\n        if self.role is Role.TOOL:\n            raise UnsupportedContentError("Tool results require ToolResultMessage")\n        content = tuple(_validate_content(block, self.role) for block in self.content)\n        return ModelMessage(self.role, content)\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelSpec:\n    model_id: str\n    context_window: int | None = None\n    max_output_tokens: int = 4096\n    supports_tools: bool = True\n\n    def __post_init__(self) -> None:\n        if not self.model_id.strip():\n            raise ValueError("ModelSpec.model_id cannot be empty")\n        if self.context_window is not None and self.context_window <= 0:\n            raise ValueError("context_window must be positive when supplied")\n        if self.max_output_tokens <= 0:\n            raise ValueError("max_output_tokens must be positive")\n\n\n@dataclass(frozen=True, slots=True)\nclass Usage:\n    input_tokens: int\n    output_tokens: int\n    total_tokens: int\n    estimated: bool = False\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolDefinition:\n    name: str\n    description: str\n    input_schema: Mapping[str, object]\n\n    def __post_init__(self) -> None:\n        object.__setattr__(self, "input_schema", MappingProxyType(dict(self.input_schema)))\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelRequest:\n    messages: tuple[ModelContextMessage, ...]\n    model: ModelSpec\n    tools: tuple[ToolDefinition, ...] = ()\n    operation: ModelOperation = ModelOperation.RUN\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelResult:\n    message: ModelMessage\n    stop_reason: StopReason\n    usage: Usage | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelError:\n    code: ModelErrorCode\n    message: str\n    retryable: bool\n    status_code: int | None = None\n    retry_after_seconds: float | None = None\n\n\nclass ModelAdapterError(RuntimeError):\n    def __init__(self, error: ModelError) -> None:\n        self.error = error\n        super().__init__(error.message)\n\n\n@dataclass(frozen=True, slots=True)\nclass TextDelta:\n    text: str\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolCallDelta:\n    index: int\n    id: str = ""\n    name: str = ""\n    arguments_delta: str = ""\n\n\n@dataclass(frozen=True, slots=True)\nclass UsageUpdate:\n    usage: Usage\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelEnd:\n    stop_reason: StopReason\n\n\nModelEvent: TypeAlias = TextDelta | ToolCallDelta | UsageUpdate | ModelEnd\n\n\nclass ModelAdapter(Protocol):\n    def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]: ...\n\n\ndef _validate_content(block: object, role: Role) -> ContentBlock:\n    if not isinstance(block, (TextContent, ToolCallContent)):\n        raise UnsupportedContentError(\n            "version one supports only TextContent and ToolCallContent"\n        )\n    if getattr(block, "schema_version", None) != 1:\n        raise UnsupportedContentError("unsupported Content Block schema version")\n    if isinstance(block, ToolCallContent) and role is not Role.ASSISTANT:\n        raise UnsupportedContentError(\n            "ToolCallContent is valid only for assistant messages"\n        )\n    return block\n\n\ndef to_model_messages(messages: Sequence[object]) -> tuple[ModelContextMessage, ...]:\n    converted: list[ModelContextMessage] = []\n    for message in messages:\n        convert = getattr(message, "to_model", None)\n        if not callable(convert):\n            raise TypeError("conversation messages must provide to_model()")\n        converted.append(convert())\n    return tuple(converted)\n\n\nclass ScriptedModelAdapter:\n    """Replay one or more provider-neutral model turns."""\n\n    def __init__(\n        self,\n        events: Sequence[ModelEvent] | Sequence[Sequence[ModelEvent]],\n    ) -> None:\n        items = tuple(events)\n        if items and isinstance(items[0], (list, tuple)):\n            self._turns = tuple(tuple(turn) for turn in items)  # type: ignore[arg-type]\n        else:\n            self._turns = (items,)  # type: ignore[assignment]\n        self._requests: list[ModelRequest] = []\n\n    @property\n    def received_requests(self) -> tuple[ModelRequest, ...]:\n        return tuple(self._requests)\n\n    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:\n        turn = len(self._requests)\n        self._requests.append(request)\n        if turn >= len(self._turns):\n            raise RuntimeError("script has no model turn remaining")\n        for event in self._turns[turn]:\n            yield event\n\n\nasync def complete(\n    adapter: ModelAdapter,\n    messages: Sequence[AgentMessage],\n    model: ModelSpec,\n) -> ModelResult:\n    request = ModelRequest(to_model_messages(messages), model)\n    text_parts: list[str] = []\n    tool_drafts: dict[int, dict[str, str]] = {}\n    usage: Usage | None = None\n    end: ModelEnd | None = None\n    async for event in adapter.stream(request):\n        if end is not None:\n            raise ModelProtocolError("an adapter emitted data after ModelEnd")\n        if isinstance(event, TextDelta):\n            text_parts.append(event.text)\n        elif isinstance(event, ToolCallDelta):\n            if event.index < 0:\n                raise ModelProtocolError("Tool Call indexes cannot be negative")\n            draft = tool_drafts.setdefault(\n                event.index, {"id": "", "name": "", "arguments": ""}\n            )\n            draft["id"] += event.id\n            draft["name"] += event.name\n            draft["arguments"] += event.arguments_delta\n        elif isinstance(event, UsageUpdate):\n            usage = event.usage\n        elif isinstance(event, ModelEnd):\n            end = event\n        else:\n            raise ModelProtocolError(\n                f"unsupported model event: {type(event).__name__}"\n            )\n    if end is None:\n        raise ModelProtocolError("an adapter stream must end with ModelEnd")\n    blocks: list[ContentBlock] = []\n    if text_parts:\n        blocks.append(TextContent("".join(text_parts)))\n    for index in sorted(tool_drafts):\n        draft = tool_drafts[index]\n        try:\n            blocks.append(ToolCallContent(**draft))\n        except (TypeError, ValueError) as error:\n            raise ModelProtocolError(f"incomplete Tool Call at index {index}") from error\n    return ModelResult(\n        message=ModelMessage(Role.ASSISTANT, tuple(blocks)),\n        stop_reason=end.stop_reason,\n        usage=usage,\n    )\n\n\n@dataclass(frozen=True, slots=True)\nclass OpenAICompatibleConfig:\n    base_url: str\n    api_key: str\n    headers: Mapping[str, str] = field(default_factory=dict)\n    extra_body: Mapping[str, object] = field(default_factory=dict)\n    timeout_seconds: float = 60.0\n\n    def __post_init__(self) -> None:\n        parsed = urlparse(self.base_url)\n        if parsed.scheme not in {"http", "https"} or not parsed.netloc:\n            raise ValueError("base_url must be an explicit HTTP(S) URL")\n        if not self.api_key:\n            raise ValueError("api_key must be supplied explicitly")\n        if self.timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive")\n        object.__setattr__(self, "headers", MappingProxyType(dict(self.headers)))\n        object.__setattr__(self, "extra_body", MappingProxyType(dict(self.extra_body)))\n\n\ndef _provider_message(message: ModelContextMessage) -> dict[str, object]:\n    if isinstance(message, ModelToolResultMessage):\n        return {\n            "role": "tool",\n            "tool_call_id": message.tool_call_id,\n            "content": message.content,\n        }\n    text = "".join(\n        block.text for block in message.content if isinstance(block, TextContent)\n    )\n    tool_calls = [\n        block for block in message.content if isinstance(block, ToolCallContent)\n    ]\n    encoded: dict[str, object] = {\n        "role": message.role.value,\n        "content": text or None,\n    }\n    if tool_calls:\n        encoded["tool_calls"] = [\n            {\n                "id": block.id,\n                "type": "function",\n                "function": {"name": block.name, "arguments": block.arguments},\n            }\n            for block in tool_calls\n        ]\n    return encoded\n\n\ndef _provider_tool(tool: ToolDefinition) -> dict[str, object]:\n    return {\n        "type": "function",\n        "function": {\n            "name": tool.name,\n            "description": tool.description,\n            "parameters": dict(tool.input_schema),\n        },\n    }\n\n\ndef _stop_reason(value: str | None) -> StopReason:\n    if value is None:\n        return StopReason.OTHER\n    return {\n        "stop": StopReason.COMPLETE,\n        "tool_calls": StopReason.TOOL_USE,\n        "length": StopReason.LENGTH,\n        "content_filter": StopReason.CONTENT_FILTER,\n    }.get(value, StopReason.OTHER)\n\n\ndef _normalized_error(error: Exception) -> ModelError:\n    status = getattr(error, "status_code", None)\n    raw_body = getattr(error, "body", None)\n    provider_markers: list[str] = []\n    for value in (\n        getattr(error, "code", None),\n        getattr(error, "type", None),\n        getattr(error, "message", None),\n    ):\n        if isinstance(value, str):\n            provider_markers.append(value.lower())\n    if isinstance(raw_body, dict):\n        for key in ("code", "type", "message"):\n            value = raw_body.get(key)\n            if isinstance(value, str):\n                provider_markers.append(value.lower())\n    context_overflow = status == 400 and any(\n        marker in value\n        for value in provider_markers\n        for marker in (\n            "context_length_exceeded",\n            "context_window_exceeded",\n            "maximum context length",\n            "context window is too long",\n        )\n    )\n    retry_after_seconds: float | None = None\n    response = getattr(error, "response", None)\n    headers = getattr(response, "headers", None)\n    if headers is not None:\n        raw_retry_after = headers.get("retry-after")\n        if raw_retry_after is not None:\n            try:\n                parsed_retry_after = float(raw_retry_after)\n            except (TypeError, ValueError):\n                pass\n            else:\n                if parsed_retry_after >= 0:\n                    retry_after_seconds = parsed_retry_after\n    if context_overflow:\n        code, retryable = ModelErrorCode.CONTEXT_OVERFLOW, False\n    elif isinstance(error, openai.AuthenticationError):\n        code, retryable = ModelErrorCode.AUTHENTICATION, False\n    elif isinstance(error, openai.RateLimitError) or status == 429:\n        code, retryable = ModelErrorCode.RATE_LIMIT, True\n    elif isinstance(error, openai.APITimeoutError) or status == 408:\n        code, retryable = ModelErrorCode.TIMEOUT, True\n    elif isinstance(error, openai.APIConnectionError):\n        code, retryable = ModelErrorCode.CONNECTION, True\n    elif isinstance(status, int) and status >= 500:\n        code, retryable = ModelErrorCode.SERVER, True\n    elif isinstance(error, (openai.BadRequestError, openai.NotFoundError)):\n        code, retryable = ModelErrorCode.REQUEST, False\n    else:\n        code, retryable = ModelErrorCode.PROVIDER, False\n    status_text = f" with status {status}" if status is not None else ""\n    return ModelError(\n        code=code,\n        message=f"OpenAI-compatible request failed{status_text}",\n        retryable=retryable,\n        status_code=status,\n        retry_after_seconds=retry_after_seconds,\n    )\n\n\nclass OpenAICompatibleAdapter:\n    """Translate streaming Chat Completions at the ModelAdapter seam."""\n\n    def __init__(self, config: OpenAICompatibleConfig) -> None:\n        self._config = config\n        self._client = openai.AsyncOpenAI(\n            api_key=config.api_key,\n            base_url=config.base_url.rstrip("/") + "/",\n            default_headers=dict(config.headers),\n            timeout=config.timeout_seconds,\n            max_retries=0,\n        )\n\n    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:\n        finish_reason: str | None = None\n        try:\n            response = await self._client.chat.completions.create(\n                model=request.model.model_id,\n                messages=cast(list, [_provider_message(item) for item in request.messages]),\n                max_tokens=request.model.max_output_tokens,\n                tools=cast(\n                    Any,\n                    [_provider_tool(tool) for tool in request.tools]\n                    if request.tools\n                    else openai.NOT_GIVEN,\n                ),\n                stream=True,\n                stream_options={"include_usage": True},\n                extra_body=dict(self._config.extra_body) or None,\n            )\n            async for chunk in response:\n                if chunk.usage is not None:\n                    input_tokens = chunk.usage.prompt_tokens or 0\n                    output_tokens = chunk.usage.completion_tokens or 0\n                    total_tokens = (\n                        chunk.usage.total_tokens or input_tokens + output_tokens\n                    )\n                    yield UsageUpdate(\n                        Usage(input_tokens, output_tokens, total_tokens)\n                    )\n                for choice in chunk.choices:\n                    delta = choice.delta\n                    if delta.content:\n                        yield TextDelta(delta.content)\n                    for tool_call in delta.tool_calls or ():\n                        function = tool_call.function\n                        yield ToolCallDelta(\n                            index=tool_call.index,\n                            id=tool_call.id or "",\n                            name=(function.name if function else None) or "",\n                            arguments_delta=(\n                                function.arguments if function else None\n                            )\n                            or "",\n                        )\n                    if choice.finish_reason is not None:\n                        finish_reason = choice.finish_reason\n        except openai.OpenAIError as error:\n            raise ModelAdapterError(_normalized_error(error)) from None\n        yield ModelEnd(_stop_reason(finish_reason))\n'


In [ ]:
PERSISTENCE_SOURCE = '"""Tree-structured Session persistence at settled boundaries."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom typing import BinaryIO, Protocol, TypeAlias\nfrom uuid import uuid4\n\nfrom .compaction import (\n    CompactionCheckpoint,\n    CompactionTrigger,\n    StructuredSummary,\n)\nfrom .model import AgentMessage, Role, TextContent, ToolCallContent, Usage\nfrom .tools import (\n    CompleteOutputKind,\n    CompleteOutputReference,\n    ToolErrorCode,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\n\n\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\nIdFactory: TypeAlias = Callable[[], str]\n\n\n@dataclass(frozen=True, slots=True)\nclass SchemaVersion:\n    major: int\n    minor: int = 0\n\n\nSESSION_SCHEMA_VERSION = SchemaVersion(1, 1)\n_SESSION_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]{0,127}\\Z")\n\n\nclass RecoveryCode(str, Enum):\n    INCOMPLETE_FINAL_RECORD = "incomplete_final_record"\n\n\n@dataclass(frozen=True, slots=True)\nclass RecoveryWarning:\n    code: RecoveryCode\n    message: str\n    line_number: int\n\n\nclass SessionBusyError(RuntimeError):\n    """Structured failure raised when a Session already owns its writer lease."""\n\n    code = "session_busy"\n\n    def __init__(self, session_id: str, message: str | None = None) -> None:\n        self.session_id = session_id\n        super().__init__(\n            message or f"Session {session_id!r} already has an active writer"\n        )\n\n\nclass UnsupportedSchemaVersionError(ValueError):\n    def __init__(self, found_major: int) -> None:\n        self.artifact = "Session"\n        self.found_major = found_major\n        self.supported_major = SESSION_SCHEMA_VERSION.major\n        super().__init__(\n            f"Session schema major {found_major} is unsupported; "\n            f"this reader supports major {self.supported_major}. "\n            "Preserve the original and migrate it with migrate_session_file()."\n        )\n\n\ndef _validate_record_version(record: object, line_number: int) -> Mapping[str, object]:\n    if not isinstance(record, dict):\n        raise ValueError(f"Session record at line {line_number} must be an object")\n    if record.get("schema") != "agent_harness.session":\n        raise ValueError(f"invalid Session schema at line {line_number}")\n    raw_version = record.get("schema_version")\n    if not isinstance(raw_version, dict):\n        raise ValueError(f"invalid Session schema version at line {line_number}")\n    major = raw_version.get("major")\n    minor = raw_version.get("minor")\n    if (\n        isinstance(major, bool)\n        or not isinstance(major, int)\n        or isinstance(minor, bool)\n        or not isinstance(minor, int)\n        or major < 1\n        or minor < 0\n    ):\n        raise ValueError(f"invalid Session schema version at line {line_number}")\n    if major != SESSION_SCHEMA_VERSION.major:\n        raise UnsupportedSchemaVersionError(major)\n    return record\n\n\ndef _session_id(value: str) -> str:\n    if not _SESSION_ID.fullmatch(value):\n        raise ValueError("Session id must be a safe local identifier")\n    return value\n\n\ndef _version_record() -> dict[str, int]:\n    return {\n        "major": SESSION_SCHEMA_VERSION.major,\n        "minor": SESSION_SCHEMA_VERSION.minor,\n    }\n\n\ndef _encode_message(message: ConversationMessage) -> dict[str, object]:\n    if isinstance(message, AgentMessage):\n        content: list[dict[str, object]] = []\n        for block in message.content:\n            if isinstance(block, TextContent):\n                content.append({"type": "text", "text": block.text})\n            elif isinstance(block, ToolCallContent):\n                content.append(\n                    {\n                        "type": "tool_call",\n                        "id": block.id,\n                        "name": block.name,\n                        "arguments": block.arguments,\n                    }\n                )\n            else:\n                raise TypeError(f"unsupported Content Block: {type(block).__name__}")\n        return {\n            "kind": "agent_message",\n            "role": message.role.value,\n            "content": content,\n        }\n    if isinstance(message, ToolResultMessage):\n        result = message.result\n        truncation = (\n            None\n            if result.truncation is None\n            else {\n                "original_bytes": result.truncation.original_bytes,\n                "original_lines": result.truncation.original_lines,\n                "retained_start_byte": result.truncation.retained_start_byte,\n                "retained_end_byte": result.truncation.retained_end_byte,\n                "retained_start_line": result.truncation.retained_start_line,\n                "retained_end_line": result.truncation.retained_end_line,\n                "direction": result.truncation.direction.value,\n            }\n        )\n        complete_output = (\n            None\n            if result.complete_output is None\n            else {\n                "kind": result.complete_output.kind.value,\n                "reference": result.complete_output.reference,\n                "reason": result.complete_output.reason,\n            }\n        )\n        return {\n            "kind": "tool_result",\n            "tool_call_id": message.tool_call_id,\n            "tool_name": message.tool_name,\n            "result": {\n                "content": result.content,\n                "metadata": dict(result.metadata),\n                "terminate": result.terminate,\n                "is_error": result.is_error,\n                "error_code": (\n                    None if result.error_code is None else result.error_code.value\n                ),\n                "truncation": truncation,\n                "complete_output": complete_output,\n            },\n        }\n    raise TypeError(f"unsupported Session message: {type(message).__name__}")\n\n\ndef _decode_message(record: Mapping[str, object]) -> ConversationMessage:\n    if record.get("kind") == "agent_message":\n        role = Role(str(record["role"]))\n        raw_content = record.get("content")\n        if not isinstance(raw_content, list):\n            raise ValueError("Session AgentMessage content must be a list")\n        blocks: list[TextContent | ToolCallContent] = []\n        for raw_block in raw_content:\n            if not isinstance(raw_block, dict):\n                raise ValueError("Session Content Block must be an object")\n            if raw_block.get("type") == "text":\n                blocks.append(TextContent(str(raw_block["text"])))\n            elif raw_block.get("type") == "tool_call":\n                blocks.append(\n                    ToolCallContent(\n                        str(raw_block["id"]),\n                        str(raw_block["name"]),\n                        str(raw_block["arguments"]),\n                    )\n                )\n            else:\n                raise ValueError("unsupported Session Content Block type")\n        return AgentMessage(role, tuple(blocks))\n    if record.get("kind") == "tool_result":\n        raw_result = record.get("result")\n        if not isinstance(raw_result, dict):\n            raise ValueError("Session ToolResult must be an object")\n        raw_truncation = raw_result.get("truncation")\n        truncation = None\n        if isinstance(raw_truncation, dict):\n            truncation = TruncationNotice(\n                int(raw_truncation["original_bytes"]),\n                int(raw_truncation["original_lines"]),\n                int(raw_truncation["retained_start_byte"]),\n                int(raw_truncation["retained_end_byte"]),\n                int(raw_truncation["retained_start_line"]),\n                int(raw_truncation["retained_end_line"]),\n                TruncationDirection(str(raw_truncation["direction"])),\n            )\n        raw_complete = raw_result.get("complete_output")\n        complete_output = None\n        if isinstance(raw_complete, dict):\n            complete_output = CompleteOutputReference(\n                CompleteOutputKind(str(raw_complete["kind"])),\n                None\n                if raw_complete.get("reference") is None\n                else str(raw_complete["reference"]),\n                None\n                if raw_complete.get("reason") is None\n                else str(raw_complete["reason"]),\n            )\n        raw_metadata = raw_result.get("metadata", {})\n        if not isinstance(raw_metadata, dict):\n            raise ValueError("Session ToolResult metadata must be an object")\n        raw_error_code = raw_result.get("error_code")\n        result = ToolResult(\n            str(raw_result["content"]),\n            metadata=raw_metadata,\n            terminate=bool(raw_result.get("terminate", False)),\n            is_error=bool(raw_result.get("is_error", False)),\n            error_code=(\n                None\n                if raw_error_code is None\n                else ToolErrorCode(str(raw_error_code))\n            ),\n            truncation=truncation,\n            complete_output=complete_output,\n        )\n        return ToolResultMessage(\n            str(record["tool_call_id"]), str(record["tool_name"]), result\n        )\n    raise ValueError("unsupported Session message kind")\n\n\ndef _encode_compaction(checkpoint: CompactionCheckpoint) -> dict[str, object]:\n    usage = checkpoint.summary_usage\n    return {\n        "trigger": checkpoint.trigger.value,\n        "summary": {\n            "schema_version": checkpoint.summary.schema_version,\n            "text": checkpoint.summary.text,\n            "focus": checkpoint.summary.focus,\n        },\n        "tokens_before": checkpoint.tokens_before,\n        "summary_usage": (\n            None\n            if usage is None\n            else {\n                "input_tokens": usage.input_tokens,\n                "output_tokens": usage.output_tokens,\n                "total_tokens": usage.total_tokens,\n                "estimated": usage.estimated,\n            }\n        ),\n        "retained_tail": [\n            _encode_message(message) for message in checkpoint.retained_tail\n        ],\n    }\n\n\ndef _decode_compaction(record: Mapping[str, object]) -> CompactionCheckpoint:\n    raw_summary = record.get("summary")\n    raw_tail = record.get("retained_tail")\n    if not isinstance(raw_summary, dict) or not isinstance(raw_tail, list):\n        raise ValueError("invalid Compaction checkpoint")\n    raw_usage = record.get("summary_usage")\n    usage = None\n    if isinstance(raw_usage, dict):\n        usage = Usage(\n            int(raw_usage["input_tokens"]),\n            int(raw_usage["output_tokens"]),\n            int(raw_usage["total_tokens"]),\n            bool(raw_usage.get("estimated", False)),\n        )\n    return CompactionCheckpoint(\n        trigger=CompactionTrigger(str(record["trigger"])),\n        summary=StructuredSummary(\n            text=str(raw_summary["text"]),\n            focus=(\n                None\n                if raw_summary.get("focus") is None\n                else str(raw_summary["focus"])\n            ),\n            schema_version=int(raw_summary.get("schema_version", 1)),\n        ),\n        tokens_before=int(str(record["tokens_before"])),\n        summary_usage=usage,\n        retained_tail=tuple(_decode_message(message) for message in raw_tail),\n    )\n\n\nclass SessionWriter(Protocol):\n    session_id: str\n\n    def __enter__(self) -> "SessionWriter": ...\n\n    def __exit__(self, *exc_info: object) -> None: ...\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> "SessionEntry": ...\n\n    def append_compaction(\n        self,\n        checkpoint: CompactionCheckpoint,\n        *,\n        parent_id: str | None = None,\n    ) -> "SessionEntry": ...\n\n\nclass SessionStore(Protocol):\n    def create(self, session_id: str | None = None) -> "SessionState": ...\n\n    def read(self, session_id: str) -> "SessionState": ...\n\n    def writer(self, session_id: str) -> SessionWriter: ...\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionEntry:\n    entry_id: str\n    parent_id: str | None\n    messages: tuple[ConversationMessage, ...] = ()\n    compaction: CompactionCheckpoint | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionState:\n    session_id: str\n    entries: tuple[SessionEntry, ...] = ()\n    recovery_warning: RecoveryWarning | None = None\n\n    def __post_init__(self) -> None:\n        _session_id(self.session_id)\n        earlier: set[str] = set()\n        for entry in self.entries:\n            if not _SESSION_ID.fullmatch(entry.entry_id):\n                raise ValueError("Session entry id must be a safe local identifier")\n            if entry.entry_id in earlier:\n                raise ValueError(f"duplicate Session entry id: {entry.entry_id}")\n            if entry.parent_id is not None and entry.parent_id not in earlier:\n                raise ValueError(\n                    "Session entry parent must reference an earlier Session entry"\n                )\n            if bool(entry.messages) == (entry.compaction is not None):\n                raise ValueError(\n                    "a Session entry requires messages or one Compaction checkpoint"\n                )\n            earlier.add(entry.entry_id)\n\n    @property\n    def active_leaf_id(self) -> str | None:\n        return self.entries[-1].entry_id if self.entries else None\n\n    def history(self, leaf_id: str | None = None) -> tuple[ConversationMessage, ...]:\n        if not self.entries:\n            if leaf_id is not None:\n                raise KeyError(f"unknown Session entry: {leaf_id}")\n            return ()\n        by_id = {entry.entry_id: entry for entry in self.entries}\n        cursor = self.active_leaf_id if leaf_id is None else leaf_id\n        path: list[SessionEntry] = []\n        while cursor is not None:\n            try:\n                entry = by_id[cursor]\n            except KeyError:\n                raise KeyError(f"unknown Session entry: {cursor}") from None\n            path.append(entry)\n            cursor = entry.parent_id\n        return tuple(\n            message for entry in reversed(path) for message in entry.messages\n        )\n\n    def compactions(\n        self, leaf_id: str | None = None\n    ) -> tuple[CompactionCheckpoint, ...]:\n        return tuple(\n            entry.compaction\n            for entry in self._path(leaf_id)\n            if entry.compaction is not None\n        )\n\n    def effective_history(\n        self, leaf_id: str | None = None\n    ) -> tuple[ConversationMessage, ...]:\n        path = self._path(leaf_id)\n        latest = next(\n            (\n                index\n                for index in range(len(path) - 1, -1, -1)\n                if path[index].compaction is not None\n            ),\n            None,\n        )\n        if latest is None:\n            return tuple(message for entry in path for message in entry.messages)\n        checkpoint = path[latest].compaction\n        assert checkpoint is not None\n        return (\n            checkpoint.summary.as_message(),\n            *checkpoint.retained_tail,\n            *(\n                message\n                for entry in path[latest + 1 :]\n                for message in entry.messages\n            ),\n        )\n\n    def _path(self, leaf_id: str | None = None) -> tuple[SessionEntry, ...]:\n        if not self.entries:\n            if leaf_id is not None:\n                raise KeyError(f"unknown Session entry: {leaf_id}")\n            return ()\n        by_id = {entry.entry_id: entry for entry in self.entries}\n        cursor = self.active_leaf_id if leaf_id is None else leaf_id\n        path: list[SessionEntry] = []\n        while cursor is not None:\n            try:\n                entry = by_id[cursor]\n            except KeyError:\n                raise KeyError(f"unknown Session entry: {cursor}") from None\n            path.append(entry)\n            cursor = entry.parent_id\n        return tuple(reversed(path))\n\n\n@dataclass(frozen=True, slots=True)\nclass MigrationResult:\n    source: Path\n    destination: Path\n    session_id: str\n    entries: int\n\n\nclass MemorySessionWriter:\n    def __init__(self, store: "MemorySessionStore", session_id: str) -> None:\n        self._store = store\n        self.session_id = session_id\n\n    def __enter__(self) -> "MemorySessionWriter":\n        return self\n\n    def __exit__(self, *exc_info: object) -> None:\n        return None\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("a settled Session entry requires messages")\n        entry = SessionEntry(self._store._id_factory(), parent, accepted)\n        self._store._sessions[self.session_id] = SessionState(\n            self.session_id, (*state.entries, entry)\n        )\n        return entry\n\n    def append_compaction(\n        self,\n        checkpoint: CompactionCheckpoint,\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        entry = SessionEntry(\n            self._store._id_factory(),\n            parent,\n            compaction=checkpoint,\n        )\n        self._store._sessions[self.session_id] = SessionState(\n            self.session_id, (*state.entries, entry)\n        )\n        return entry\n\n\nclass MemorySessionStore:\n    """In-process SessionStore adapter with the durable tree contract."""\n\n    def __init__(self, *, id_factory: IdFactory | None = None) -> None:\n        self._id_factory = id_factory or (lambda: uuid4().hex)\n        self._sessions: dict[str, SessionState] = {}\n\n    def create(self, session_id: str | None = None) -> SessionState:\n        accepted = _session_id(session_id or self._id_factory())\n        if accepted in self._sessions:\n            raise ValueError(f"Session already exists: {accepted}")\n        state = SessionState(accepted)\n        self._sessions[accepted] = state\n        return state\n\n    def read(self, session_id: str) -> SessionState:\n        try:\n            return self._sessions[session_id]\n        except KeyError:\n            raise KeyError(f"unknown Session: {session_id}") from None\n\n    def writer(self, session_id: str) -> MemorySessionWriter:\n        self.read(session_id)\n        return MemorySessionWriter(self, session_id)\n\n\nclass JSONLSessionWriter:\n    def __init__(self, store: "JSONLSessionStore", session_id: str) -> None:\n        self._store = store\n        self.session_id = session_id\n        self._lock_stream: BinaryIO | None = None\n\n    def __enter__(self) -> "JSONLSessionWriter":\n        if self._lock_stream is not None:\n            raise RuntimeError("Session writer lease is already active")\n        self._lock_stream = self._store._acquire_lock(self.session_id)\n        try:\n            state = self._store.read(self.session_id)\n            if state.recovery_warning is not None:\n                self._store._discard_uncommitted_tail(self.session_id)\n        except BaseException:\n            self._store._release_lock(self._lock_stream)\n            self._lock_stream = None\n            raise\n        return self\n\n    def __exit__(self, *exc_info: object) -> None:\n        if self._lock_stream is not None:\n            self._store._release_lock(self._lock_stream)\n            self._lock_stream = None\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        if self._lock_stream is None:\n            raise RuntimeError("Session writer lease is not active")\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("a settled Session entry requires messages")\n        entry = SessionEntry(self._store._id_factory(), parent, accepted)\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "settlement",\n            "session_id": self.session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n            "messages": [_encode_message(message) for message in accepted],\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        with self._store.path_for(self.session_id).open("a", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        return entry\n\n    def append_compaction(\n        self,\n        checkpoint: CompactionCheckpoint,\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        if self._lock_stream is None:\n            raise RuntimeError("Session writer lease is not active")\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        entry = SessionEntry(\n            self._store._id_factory(),\n            parent,\n            compaction=checkpoint,\n        )\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "compaction",\n            "session_id": self.session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n            "checkpoint": _encode_compaction(checkpoint),\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        with self._store.path_for(self.session_id).open("a", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        return entry\n\n\nclass JSONLSessionStore:\n    """Transparent file-per-Session JSONL adapter."""\n\n    def __init__(self, root: str | Path, *, id_factory: IdFactory | None = None) -> None:\n        self.root = Path(root)\n        self.sessions_directory = self.root / "sessions"\n        self._id_factory = id_factory or (lambda: uuid4().hex)\n\n    def path_for(self, session_id: str) -> Path:\n        return self.sessions_directory / f"{_session_id(session_id)}.jsonl"\n\n    def _lock_path(self, session_id: str) -> Path:\n        return self.sessions_directory / ".locks" / f"{_session_id(session_id)}.lock"\n\n    def _discard_uncommitted_tail(self, session_id: str) -> None:\n        path = self.path_for(session_id)\n        content = path.read_bytes()\n        committed_end = content.rfind(b"\\n")\n        if committed_end < 0:\n            raise ValueError("Session file has no committed header")\n        with path.open("r+b") as stream:\n            stream.truncate(committed_end + 1)\n            stream.flush()\n            os.fsync(stream.fileno())\n\n    def _acquire_lock(self, session_id: str) -> BinaryIO:\n        path = self._lock_path(session_id)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        stream = path.open("a+b")\n        try:\n            if os.name == "nt":\n                import msvcrt\n\n                stream.seek(0, os.SEEK_END)\n                if stream.tell() == 0:\n                    stream.write(b"\\0")\n                    stream.flush()\n                stream.seek(0)\n                msvcrt.locking(  # type: ignore[attr-defined]\n                    stream.fileno(), msvcrt.LK_NBLCK, 1  # type: ignore[attr-defined]\n                )\n            else:\n                import fcntl\n\n                fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except OSError:\n            stream.close()\n            raise SessionBusyError(session_id) from None\n        return stream\n\n    @staticmethod\n    def _release_lock(stream: BinaryIO) -> None:\n        try:\n            if os.name == "nt":\n                import msvcrt\n\n                stream.seek(0)\n                msvcrt.locking(  # type: ignore[attr-defined]\n                    stream.fileno(), msvcrt.LK_UNLCK, 1  # type: ignore[attr-defined]\n                )\n            else:\n                import fcntl\n\n                fcntl.flock(stream.fileno(), fcntl.LOCK_UN)\n        finally:\n            stream.close()\n\n    def create(self, session_id: str | None = None) -> SessionState:\n        accepted = _session_id(session_id or self._id_factory())\n        path = self.path_for(accepted)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "session",\n            "session_id": accepted,\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        try:\n            with path.open("x", encoding="utf-8") as stream:\n                stream.write(encoded)\n                stream.flush()\n                os.fsync(stream.fileno())\n        except FileExistsError:\n            raise ValueError(f"Session already exists: {accepted}") from None\n        return SessionState(accepted)\n\n    def read(self, session_id: str) -> SessionState:\n        accepted = _session_id(session_id)\n        path = self.path_for(accepted)\n        try:\n            content = path.read_bytes()\n        except FileNotFoundError:\n            raise KeyError(f"unknown Session: {accepted}") from None\n        raw_lines = content.splitlines(keepends=True)\n        warning = None\n        if raw_lines and not raw_lines[-1].endswith(b"\\n"):\n            warning = RecoveryWarning(\n                RecoveryCode.INCOMPLETE_FINAL_RECORD,\n                "ignored an incomplete final Session record; restart the operation",\n                len(raw_lines),\n            )\n            raw_lines = raw_lines[:-1]\n        try:\n            lines = [line.decode("utf-8").rstrip("\\r\\n") for line in raw_lines]\n        except UnicodeDecodeError as error:\n            raise ValueError("Session JSONL must be UTF-8 text") from error\n        if not lines:\n            raise ValueError("Session file has no committed header")\n        entries: list[SessionEntry] = []\n        for line_number, line in enumerate(lines, 1):\n            try:\n                decoded = json.loads(line)\n            except json.JSONDecodeError as error:\n                raise ValueError(\n                    f"invalid Session JSONL record at line {line_number}"\n                ) from error\n            record = _validate_record_version(decoded, line_number)\n            if line_number == 1:\n                if record.get("record") != "session" or record.get("session_id") != accepted:\n                    raise ValueError("invalid Session header")\n                continue\n            if record.get("session_id") != accepted:\n                raise ValueError(f"invalid Session record at line {line_number}")\n            if record.get("record") == "compaction":\n                raw_checkpoint = record.get("checkpoint")\n                if not isinstance(raw_checkpoint, dict):\n                    raise ValueError("invalid Compaction checkpoint record")\n                entries.append(\n                    SessionEntry(\n                        str(record["entry_id"]),\n                        (\n                            None\n                            if record.get("parent_id") is None\n                            else str(record["parent_id"])\n                        ),\n                        compaction=_decode_compaction(raw_checkpoint),\n                    )\n                )\n                continue\n            if record.get("record") != "settlement":\n                raise ValueError(f"invalid Session record at line {line_number}")\n            raw_messages = record.get("messages")\n            if not isinstance(raw_messages, list) or not raw_messages:\n                raise ValueError("a settled Session entry requires messages")\n            entries.append(\n                SessionEntry(\n                    str(record["entry_id"]),\n                    None if record.get("parent_id") is None else str(record["parent_id"]),\n                    tuple(_decode_message(message) for message in raw_messages),\n                )\n            )\n        state = SessionState(accepted, tuple(entries), warning)\n        state.history()\n        return state\n\n    def writer(self, session_id: str) -> JSONLSessionWriter:\n        self.read(session_id)\n        return JSONLSessionWriter(self, session_id)\n\n\ndef migrate_session_file(\n    source: str | Path,\n    destination: str | Path,\n) -> MigrationResult:\n    """Write and validate a current Session file without changing its source."""\n\n    source_path = Path(source).resolve()\n    destination_path = Path(destination).resolve()\n    if source_path.parent.name != "sessions":\n        raise ValueError("source must be a file from a sessions directory")\n    session_id = _session_id(source_path.stem)\n    if destination_path.name != f"{session_id}.jsonl":\n        raise ValueError("migration destination must retain the Session filename")\n    if destination_path.exists():\n        raise FileExistsError(f"migration destination exists: {destination_path}")\n    state = JSONLSessionStore(source_path.parent.parent).read(session_id)\n    records: list[dict[str, object]] = [\n        {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "session",\n            "session_id": session_id,\n        }\n    ]\n    records.extend(\n        (\n            {\n                "schema": "agent_harness.session",\n                "schema_version": _version_record(),\n                "record": "settlement",\n                "session_id": session_id,\n                "entry_id": entry.entry_id,\n                "parent_id": entry.parent_id,\n                "messages": [_encode_message(message) for message in entry.messages],\n            }\n            if entry.compaction is None\n            else {\n                "schema": "agent_harness.session",\n                "schema_version": _version_record(),\n                "record": "compaction",\n                "session_id": session_id,\n                "entry_id": entry.entry_id,\n                "parent_id": entry.parent_id,\n                "checkpoint": _encode_compaction(entry.compaction),\n            }\n        )\n        for entry in state.entries\n    )\n    encoded = "".join(\n        json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n        + "\\n"\n        for record in records\n    )\n    destination_path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(\n        prefix=".session-migration-", dir=destination_path.parent.parent\n    ) as temporary:\n        staging_root = Path(temporary)\n        staging = staging_root / "sessions" / f"{session_id}.jsonl"\n        staging.parent.mkdir(parents=True)\n        with staging.open("x", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        validated = JSONLSessionStore(staging_root).read(session_id)\n        if (\n            validated.history() != state.history()\n            or validated.compactions() != state.compactions()\n        ):\n            raise ValueError("migrated Session failed history validation")\n        staging.replace(destination_path)\n    return MigrationResult(\n        source_path,\n        destination_path,\n        session_id,\n        len(state.entries),\n    )\n'


In [ ]:
RUNTIME_SOURCE = '"""Async Agent Runtime with structured Tool batches."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import AsyncIterator, Awaitable, Callable, Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nimport json\nfrom typing import Protocol, TypeAlias, cast\n\nfrom jsonschema import (  # type: ignore[import-untyped]\n    Draft202012Validator,\n    ValidationError,\n)\n\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelRequest,\n    ModelOperation,\n    ModelSpec,\n    Role,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    Usage,\n    UsageUpdate,\n    to_model_messages,\n)\nfrom .compaction import CompactionCheckpoint\nfrom .tools import (\n    LocalToolExecutor,\n    PreparedToolCall,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    bound_tool_result,\n)\n\n\nclass EventType(str, Enum):\n    AGENT_START = "agent_start"\n    MODEL_ATTEMPT_START = "model_attempt_start"\n    MODEL_EVENT = "model_event"\n    MODEL_ATTEMPT_FAILED = "model_attempt_failed"\n    RETRY_SCHEDULED = "retry_scheduled"\n    COMPACTION_START = "compaction_start"\n    COMPACTION_END = "compaction_end"\n    COMPACTION_FAILED = "compaction_failed"\n    TOOL_BATCH_START = "tool_batch_start"\n    TOOL_CALL_START = "tool_call_start"\n    TOOL_CALL_END = "tool_call_end"\n    TOOL_BATCH_END = "tool_batch_end"\n    RUN_CANCELLED = "run_cancelled"\n    MESSAGE_END = "message_end"\n    AGENT_END = "agent_end"\n\n\nclass TerminalStatus(str, Enum):\n    COMPLETED = "completed"\n    MODEL_ERROR = "model_error"\n    CANCELLED = "cancelled"\n    MAX_TURNS = "max_turns"\n    MAX_TOOL_CALLS = "max_tool_calls"\n    TIMEOUT = "timeout"\n    MAX_TOTAL_TOKENS = "max_total_tokens"\n\n\n@dataclass(frozen=True, slots=True)\nclass RunGuard:\n    max_turns: int | None = None\n    max_tool_calls: int | None = None\n    timeout_seconds: float | None = None\n    max_total_tokens: int | None = None\n\n    def __post_init__(self) -> None:\n        integer_limits = {\n            "max_turns": self.max_turns,\n            "max_tool_calls": self.max_tool_calls,\n            "max_total_tokens": self.max_total_tokens,\n        }\n        for name, value in integer_limits.items():\n            if value is not None and (\n                isinstance(value, bool) or not isinstance(value, int) or value <= 0\n            ):\n                raise ValueError(f"{name} must be a positive integer when supplied")\n        if self.timeout_seconds is not None and self.timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive when supplied")\n\n    def reached(\n        self,\n        *,\n        turns: int,\n        tool_calls: int,\n        usage: Usage | None,\n        elapsed_seconds: float,\n    ) -> TerminalStatus | None:\n        if (\n            self.timeout_seconds is not None\n            and elapsed_seconds >= self.timeout_seconds\n        ):\n            return TerminalStatus.TIMEOUT\n        if self.max_turns is not None and turns >= self.max_turns:\n            return TerminalStatus.MAX_TURNS\n        if (\n            self.max_tool_calls is not None\n            and tool_calls >= self.max_tool_calls\n        ):\n            return TerminalStatus.MAX_TOOL_CALLS\n        if (\n            self.max_total_tokens is not None\n            and usage is not None\n            and usage.total_tokens >= self.max_total_tokens\n        ):\n            return TerminalStatus.MAX_TOTAL_TOKENS\n        return None\n\n\n@dataclass(frozen=True, slots=True)\nclass RuntimeEvent:\n    sequence: int\n    type: EventType\n    attempt: int | None = None\n    model_event: ModelEvent | None = None\n    error: ModelError | None = None\n    retry_delay_seconds: float | None = None\n    partial_text: str = ""\n    partial_usage: Usage | None = None\n    tool_call_id: str | None = None\n    tool_name: str | None = None\n    tool_result: ToolResult | None = None\n    operation: ModelOperation = ModelOperation.RUN\n\n\n@dataclass(frozen=True, slots=True)\nclass AssistantOutcome:\n    message: AgentMessage\n    stop_reason: StopReason\n    usage: Usage | None = None\n    error: ModelError | None = None\n    attempts: int = 1\n    tool_results: tuple[ToolResult, ...] = ()\n    status: TerminalStatus = TerminalStatus.COMPLETED\n\n\n@dataclass(frozen=True, slots=True)\nclass SummaryGeneration:\n    text: str\n    usage: Usage | None\n    attempts: int\n\n\n@dataclass(frozen=True, slots=True)\nclass RetryPolicy:\n    delays: tuple[float, ...] = (2.0, 4.0, 8.0)\n    max_retry_after_seconds: float = 60.0\n\n    def __post_init__(self) -> None:\n        if any(delay < 0 for delay in self.delays):\n            raise ValueError("retry delays cannot be negative")\n        if self.max_retry_after_seconds < 0:\n            raise ValueError("max_retry_after_seconds cannot be negative")\n\n    def delay_for(\n        self,\n        error: ModelError,\n        failed_attempt: int,\n        *,\n        retry_after_seconds: float | None = None,\n    ) -> float | None:\n        retryable_codes = {\n            ModelErrorCode.RATE_LIMIT,\n            ModelErrorCode.TIMEOUT,\n            ModelErrorCode.CONNECTION,\n            ModelErrorCode.SERVER,\n        }\n        retryable_status = error.status_code in {408, 429} or (\n            error.status_code is not None and error.status_code >= 500\n        )\n        if (\n            error.code not in retryable_codes and not retryable_status\n        ) or failed_attempt > len(self.delays):\n            return None\n        if retry_after_seconds is not None:\n            if not 0 <= retry_after_seconds <= self.max_retry_after_seconds:\n                return None\n            return retry_after_seconds\n        return self.delays[failed_attempt - 1]\n\n\nSleeper: TypeAlias = Callable[[float], Awaitable[None]]\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\nSettlementSink: TypeAlias = Callable[[Sequence[ConversationMessage]], None]\nContextOverflowRecovery: TypeAlias = Callable[[], Awaitable[bool]]\nSettledTurnHandler: TypeAlias = Callable[[], Awaitable[object]]\n_EVENTS_DONE = object()\n\n\nclass TurnInput(Protocol):\n    """Runtime-facing view of Steering Messages waiting at a turn boundary."""\n\n    def pending(self) -> bool: ...\n\n    def take(self) -> Sequence[AgentMessage]: ...\n\n\n@dataclass(slots=True)\nclass _CancellationState:\n    status: TerminalStatus = TerminalStatus.CANCELLED\n    requested: bool = False\n\n    def request(self, status: TerminalStatus) -> bool:\n        if self.requested:\n            return False\n        self.status = status\n        self.requested = True\n        return True\n\n\ndef _add_usage(left: Usage | None, right: Usage | None) -> Usage | None:\n    if left is None:\n        return right\n    if right is None:\n        return left\n    return Usage(\n        left.input_tokens + right.input_tokens,\n        left.output_tokens + right.output_tokens,\n        left.total_tokens + right.total_tokens,\n        left.estimated or right.estimated,\n    )\n\n\nclass AgentRunHandle:\n    """One accepted run\'s observations, cancellation, and eventual outcome."""\n\n    def __init__(\n        self,\n        task: asyncio.Task[AssistantOutcome],\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n    ) -> None:\n        self._task = task\n        self._events = events\n        self._cancellation = cancellation\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n    async def result(self) -> AssistantOutcome:\n        return await self._task\n\n    def cancel(self) -> None:\n        self._cancel_with(TerminalStatus.CANCELLED)\n\n    def _cancel_with(self, status: TerminalStatus) -> None:\n        if not self._task.done() and self._cancellation.request(status):\n            self._task.get_loop().call_soon(self._task.cancel)\n\n\nclass AgentRuntime:\n    """Advance typed conversation state through model and Tool turns."""\n\n    def __init__(\n        self,\n        adapter: ModelAdapter,\n        model: ModelSpec,\n        *,\n        tools: Sequence[Tool] = (),\n        tool_executor: ToolExecutor | None = None,\n        tool_output_budget: ToolOutputBudget | None = None,\n        retry_policy: RetryPolicy | None = None,\n        sleeper: Sleeper = asyncio.sleep,\n        run_guard: object | None = None,\n        history: Sequence[ConversationMessage] = (),\n    ) -> None:\n        if not isinstance(model, ModelSpec):\n            raise TypeError("model must be a ModelSpec")\n        if not callable(getattr(adapter, "stream", None)):\n            raise TypeError("adapter must implement ModelAdapter.stream")\n        if retry_policy is not None and not isinstance(retry_policy, RetryPolicy):\n            raise TypeError("retry_policy must be a RetryPolicy")\n        if not callable(sleeper):\n            raise TypeError("sleeper must be an async callable")\n        if tool_executor is not None and not callable(\n            getattr(tool_executor, "execute", None)\n        ):\n            raise TypeError("tool_executor must implement ToolExecutor.execute")\n        if tool_output_budget is not None and not isinstance(\n            tool_output_budget, ToolOutputBudget\n        ):\n            raise TypeError("tool_output_budget must be a ToolOutputBudget")\n        registered: dict[str, Tool] = {}\n        for tool in tools:\n            if not isinstance(tool, Tool):\n                raise TypeError("tools must contain Tool values")\n            if tool.name in registered:\n                raise ValueError(f"duplicate Tool name: {tool.name!r}")\n            registered[tool.name] = tool\n        if registered and not model.supports_tools:\n            raise ValueError("configured ModelSpec does not support Tools")\n        self._adapter = adapter\n        self._model = model\n        self._tools = registered\n        self._tool_executor = tool_executor or LocalToolExecutor()\n        self._tool_output_budget = tool_output_budget or ToolOutputBudget()\n        self._retry_policy = retry_policy or RetryPolicy()\n        self._sleeper = sleeper\n        self._run_guard = run_guard\n        accepted_history = tuple(history)\n        to_model_messages(accepted_history)\n        self._history: list[ConversationMessage] = list(accepted_history)\n        self._effective_history: list[ConversationMessage] | None = None\n        self._settlement_sink: SettlementSink | None = None\n        self._context_overflow_recovery: ContextOverflowRecovery | None = None\n        self._settled_turn_handler: SettledTurnHandler | None = None\n\n    @property\n    def history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(self._history)\n\n    @property\n    def effective_history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(\n            self._history\n            if self._effective_history is None\n            else self._effective_history\n        )\n\n    @property\n    def model(self) -> ModelSpec:\n        return self._model\n\n    @property\n    def run_guard(self) -> object | None:\n        return self._run_guard\n\n    def restore_history(\n        self,\n        history: Sequence[ConversationMessage],\n        *,\n        effective_history: Sequence[ConversationMessage] | None = None,\n    ) -> None:\n        """Seed a newly constructed Runtime from one settled Session branch."""\n\n        if self._history:\n            raise RuntimeError("Runtime history must be empty before restoration")\n        accepted = tuple(history)\n        to_model_messages(accepted)\n        self._history.extend(accepted)\n        if effective_history is not None:\n            effective = tuple(effective_history)\n            to_model_messages(effective)\n            self._effective_history = list(effective)\n\n    def install_compaction(self, checkpoint: CompactionCheckpoint) -> None:\n        """Replace only the model-facing prefix at a Settled Boundary."""\n\n        self._effective_history = [\n            checkpoint.summary.as_message(),\n            *checkpoint.retained_tail,\n        ]\n\n    def set_settlement_sink(self, sink: SettlementSink | None) -> None:\n        """Install the AgentSession-owned persistence barrier for the next Run."""\n\n        if sink is not None and not callable(sink):\n            raise TypeError("settlement sink must be callable")\n        self._settlement_sink = sink\n\n    def set_context_overflow_recovery(\n        self,\n        recovery: ContextOverflowRecovery | None,\n    ) -> None:\n        if recovery is not None and not callable(recovery):\n            raise TypeError("context overflow recovery must be callable")\n        self._context_overflow_recovery = recovery\n\n    def set_settled_turn_handler(\n        self,\n        handler: SettledTurnHandler | None,\n    ) -> None:\n        if handler is not None and not callable(handler):\n            raise TypeError("settled turn handler must be callable")\n        self._settled_turn_handler = handler\n\n    def _append_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        self._history.extend(accepted)\n        if self._effective_history is not None:\n            self._effective_history.extend(accepted)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def _append_unsettled(self, message: ConversationMessage) -> None:\n        self._history.append(message)\n        if self._effective_history is not None:\n            self._effective_history.append(message)\n\n    def _mark_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def start(\n        self,\n        messages: Sequence[AgentMessage],\n        *,\n        turn_input: TurnInput | None = None,\n    ) -> AgentRunHandle:\n        accepted = tuple(messages)\n        to_model_messages((*self.effective_history, *accepted))\n        self._append_settled(accepted)\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        cancellation = _CancellationState()\n        task = asyncio.get_running_loop().create_task(\n            self._execute(events, cancellation, turn_input)\n        )\n        handle = AgentRunHandle(task, events, cancellation)\n        if (\n            isinstance(self._run_guard, RunGuard)\n            and self._run_guard.timeout_seconds is not None\n        ):\n            timer = asyncio.get_running_loop().call_later(\n                self._run_guard.timeout_seconds,\n                handle._cancel_with,\n                TerminalStatus.TIMEOUT,\n            )\n            task.add_done_callback(lambda completed: timer.cancel())\n        return handle\n\n    async def run(self, messages: Sequence[AgentMessage]) -> AssistantOutcome:\n        return await self.start(messages).result()\n\n    async def generate_summary(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        focus: str | None = None,\n    ) -> SummaryGeneration:\n        """Run a retryable model operation without mutating conversation history."""\n\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("summary generation requires source history")\n        instruction = (\n            "Summarize the following settled conversation as a durable context "\n            "checkpoint. Preserve decisions, constraints, unresolved work, and "\n            "facts needed to continue. Return summary text only."\n        )\n        if focus is not None:\n            instruction += f"\\nFocus requested by the caller: {focus}"\n        summary_prompt = AgentMessage.text(Role.SYSTEM, instruction)\n        attempt = 0\n        while True:\n            attempt += 1\n            request = ModelRequest(\n                messages=to_model_messages((summary_prompt, *accepted)),\n                model=self._model,\n                operation=ModelOperation.COMPACTION,\n            )\n            text_parts: list[str] = []\n            usage: Usage | None = None\n            end: ModelEnd | None = None\n            schema_error: ModelError | None = None\n            try:\n                async for event in self._adapter.stream(request):\n                    if end is not None:\n                        schema_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "Compaction stream emitted data after ModelEnd",\n                            False,\n                        )\n                        break\n                    if isinstance(event, TextDelta):\n                        text_parts.append(event.text)\n                    elif isinstance(event, UsageUpdate):\n                        usage = event.usage\n                    elif isinstance(event, ModelEnd):\n                        end = event\n                    else:\n                        schema_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "Compaction summary must contain text only",\n                            False,\n                        )\n                        break\n            except ModelAdapterError as failure:\n                delay = self._retry_policy.delay_for(\n                    failure.error,\n                    attempt,\n                    retry_after_seconds=failure.error.retry_after_seconds,\n                )\n                if delay is None:\n                    raise\n                await self._sleeper(delay)\n                continue\n            if schema_error is None and end is None:\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction stream did not end with ModelEnd",\n                    False,\n                )\n            if (\n                schema_error is None\n                and end is not None\n                and end.stop_reason is not StopReason.COMPLETE\n            ):\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction summary did not complete successfully",\n                    False,\n                )\n            text = "".join(text_parts)\n            if schema_error is None and not text.strip():\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction summary cannot be empty",\n                    False,\n                )\n            if schema_error is not None:\n                raise ModelAdapterError(schema_error)\n            return SummaryGeneration(text, usage, attempt)\n\n    async def _execute(\n        self,\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n        turn_input: TurnInput | None = None,\n    ) -> AssistantOutcome:\n        sequence = 0\n        total_attempts = 0\n        text_parts: list[str] = []\n        current_usage: Usage | None = None\n        run_usage: Usage | None = None\n        run_tool_results: list[ToolResult] = []\n        turns = 0\n        tool_calls = 0\n        started_at = asyncio.get_running_loop().time()\n        overflow_recovery_attempted = False\n\n        def guard_status() -> TerminalStatus | None:\n            if not isinstance(self._run_guard, RunGuard):\n                return None\n            return self._run_guard.reached(\n                turns=turns,\n                tool_calls=tool_calls,\n                usage=run_usage,\n                elapsed_seconds=asyncio.get_running_loop().time() - started_at,\n            )\n\n        async def emit(\n            type_: EventType,\n            *,\n            attempt: int | None = None,\n            model_event: ModelEvent | None = None,\n            error: ModelError | None = None,\n            retry_delay_seconds: float | None = None,\n            partial_text: str = "",\n            partial_usage: Usage | None = None,\n            tool_call_id: str | None = None,\n            tool_name: str | None = None,\n            tool_result: ToolResult | None = None,\n            operation: ModelOperation = ModelOperation.RUN,\n        ) -> None:\n            nonlocal sequence\n            sequence += 1\n            await events.put(\n                RuntimeEvent(\n                    sequence=sequence,\n                    type=type_,\n                    attempt=attempt,\n                    model_event=model_event,\n                    error=error,\n                    retry_delay_seconds=retry_delay_seconds,\n                    partial_text=partial_text,\n                    partial_usage=partial_usage,\n                    tool_call_id=tool_call_id,\n                    tool_name=tool_name,\n                    tool_result=tool_result,\n                    operation=operation,\n                )\n            )\n\n        async def finish(outcome: AssistantOutcome) -> AssistantOutcome:\n            self._append_settled((outcome.message,))\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n\n        try:\n            await emit(EventType.AGENT_START)\n            while True:\n                turn_attempt = 0\n                while True:\n                    turn_attempt += 1\n                    total_attempts += 1\n                    attempt = total_attempts\n                    request = ModelRequest(\n                        to_model_messages(self.effective_history),\n                        self._model,\n                        tuple(tool.definition() for tool in self._tools.values()),\n                    )\n                    await emit(EventType.MODEL_ATTEMPT_START, attempt=attempt)\n                    text_parts = []\n                    tool_drafts: dict[int, dict[str, str]] = {}\n                    current_usage = None\n                    end: ModelEnd | None = None\n                    schema_error: ModelError | None = None\n                    try:\n                        async for event in self._adapter.stream(request):\n                            await emit(\n                                EventType.MODEL_EVENT,\n                                attempt=attempt,\n                                model_event=event,\n                            )\n                            if end is not None:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted data after ModelEnd",\n                                    False,\n                                )\n                                break\n                            if isinstance(event, TextDelta):\n                                text_parts.append(event.text)\n                            elif isinstance(event, ToolCallDelta):\n                                if event.index < 0:\n                                    schema_error = ModelError(\n                                        ModelErrorCode.SCHEMA,\n                                        "model stream emitted an invalid Tool Call index",\n                                        False,\n                                    )\n                                    break\n                                draft = tool_drafts.setdefault(\n                                    event.index,\n                                    {"id": "", "name": "", "arguments": ""},\n                                )\n                                draft["id"] += event.id\n                                draft["name"] += event.name\n                                draft["arguments"] += event.arguments_delta\n                            elif isinstance(event, UsageUpdate):\n                                current_usage = event.usage\n                            elif isinstance(event, ModelEnd):\n                                end = event\n                            else:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted an unsupported event",\n                                    False,\n                                )\n                                break\n                    except ModelAdapterError as failure:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=failure.error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        if (\n                            failure.error.code is ModelErrorCode.CONTEXT_OVERFLOW\n                            and not overflow_recovery_attempted\n                            and self._context_overflow_recovery is not None\n                        ):\n                            overflow_recovery_attempted = True\n                            await emit(\n                                EventType.COMPACTION_START,\n                                attempt=attempt,\n                                error=failure.error,\n                                operation=ModelOperation.COMPACTION,\n                            )\n                            try:\n                                recovered = await self._context_overflow_recovery()\n                            except Exception as recovery_error:\n                                recovered = False\n                                failure = ModelAdapterError(\n                                    ModelError(\n                                        ModelErrorCode.COMPACTION_FAILED,\n                                        "context overflow recovery Compaction failed: "\n                                        f"{type(recovery_error).__name__}",\n                                        False,\n                                    )\n                                )\n                            await emit(\n                                (\n                                    EventType.COMPACTION_END\n                                    if recovered\n                                    else EventType.COMPACTION_FAILED\n                                ),\n                                attempt=attempt,\n                                error=None if recovered else failure.error,\n                                operation=ModelOperation.COMPACTION,\n                            )\n                            if recovered:\n                                text_parts = []\n                                current_usage = None\n                                continue\n                        delay = self._retry_policy.delay_for(\n                            failure.error,\n                            turn_attempt,\n                            retry_after_seconds=failure.error.retry_after_seconds,\n                        )\n                        if delay is not None:\n                            await emit(\n                                EventType.RETRY_SCHEDULED,\n                                attempt=attempt,\n                                error=failure.error,\n                                retry_delay_seconds=delay,\n                                partial_text=partial_text,\n                                partial_usage=current_usage,\n                            )\n                            text_parts = []\n                            current_usage = None\n                            await self._sleeper(delay)\n                            continue\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                failure.error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n\n                    error = schema_error\n                    if error is None and end is None:\n                        error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "model stream violated the provider-neutral event contract",\n                            False,\n                        )\n                    blocks: list[ContentBlock] = []\n                    if error is None:\n                        try:\n                            if text_parts:\n                                blocks.append(TextContent("".join(text_parts)))\n                            for index in sorted(tool_drafts):\n                                blocks.append(ToolCallContent(**tool_drafts[index]))\n                        except (TypeError, ValueError):\n                            error = ModelError(\n                                ModelErrorCode.SCHEMA,\n                                "model stream emitted an incomplete Tool Call",\n                                False,\n                            )\n                    if error is not None:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n                    assert end is not None\n                    break\n\n                run_usage = _add_usage(run_usage, current_usage)\n                turns += 1\n                assistant = AgentMessage(Role.ASSISTANT, tuple(blocks))\n                calls = tuple(\n                    block\n                    for block in assistant.content\n                    if isinstance(block, ToolCallContent)\n                )\n                if calls:\n                    self._append_unsettled(assistant)\n                else:\n                    self._append_settled((assistant,))\n                await emit(EventType.MESSAGE_END, attempt=total_attempts)\n                if not calls:\n                    if self._settled_turn_handler is not None:\n                        await self._settled_turn_handler()\n                    if turn_input is not None and turn_input.pending():\n                        reached = guard_status()\n                        if reached is not None:\n                            await emit(EventType.AGENT_END, attempt=total_attempts)\n                            return AssistantOutcome(\n                                assistant,\n                                StopReason.ABORTED,\n                                run_usage,\n                                attempts=total_attempts,\n                                tool_results=tuple(run_tool_results),\n                                status=reached,\n                            )\n                        steering = tuple(turn_input.take())\n                        to_model_messages(steering)\n                        self._append_settled(steering)\n                        continue\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n\n                await emit(EventType.TOOL_BATCH_START, attempt=total_attempts)\n                prepared: dict[int, PreparedToolCall] = {}\n                results: dict[int, ToolResult] = {}\n                for index, call in enumerate(calls):\n                    tool = self._tools.get(call.name)\n                    if tool is None:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.UNKNOWN_TOOL, call.name\n                        )\n                        continue\n                    try:\n                        parsed = json.loads(call.arguments)\n                    except (json.JSONDecodeError, TypeError):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_JSON, call.name\n                        )\n                        continue\n                    try:\n                        Draft202012Validator(tool.input_schema).validate(parsed)\n                    except ValidationError:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    if not isinstance(parsed, dict):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    prepared[index] = PreparedToolCall(call, tool, parsed)\n\n                async def execute_one(index: int, call: PreparedToolCall) -> None:\n                    await emit(\n                        EventType.TOOL_CALL_START,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                    )\n                    try:\n                        result = await self._tool_executor.execute(call)\n                        if not isinstance(result, ToolResult):\n                            raise TypeError("ToolExecutor returned an invalid result")\n                    except Exception:\n                        result = ToolResult.error(\n                            ToolErrorCode.EXECUTION_FAILED, call.call.name\n                        )\n                    result = bound_tool_result(\n                        result,\n                        self._tool_output_budget,\n                        call.tool.output_direction,\n                    )\n                    results[index] = result\n                    await emit(\n                        EventType.TOOL_CALL_END,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                        tool_result=result,\n                    )\n\n                try:\n                    if any(call.tool.sequential for call in prepared.values()):\n                        for index, prepared_call in prepared.items():\n                            await execute_one(index, prepared_call)\n                    else:\n                        tasks = {\n                            index: asyncio.create_task(\n                                execute_one(index, prepared_call)\n                            )\n                            for index, prepared_call in prepared.items()\n                        }\n                        try:\n                            await asyncio.gather(*tasks.values())\n                        except asyncio.CancelledError:\n                            for task in tasks.values():\n                                if not task.done():\n                                    task.cancel()\n                            await asyncio.gather(\n                                *tasks.values(), return_exceptions=True\n                            )\n                            raise\n                except asyncio.CancelledError:\n                    for index, prepared_call in prepared.items():\n                        if index not in results:\n                            cancelled_result = ToolResult.error(\n                                ToolErrorCode.CANCELLED,\n                                prepared_call.call.name,\n                            )\n                            results[index] = cancelled_result\n                            await emit(\n                                EventType.TOOL_CALL_END,\n                                attempt=total_attempts,\n                                tool_call_id=prepared_call.call.id,\n                                tool_name=prepared_call.call.name,\n                                tool_result=cancelled_result,\n                            )\n                    for index, call in enumerate(calls):\n                        result = results[index]\n                        run_tool_results.append(result)\n                        self._append_unsettled(\n                            ToolResultMessage(call.id, call.name, result)\n                        )\n                    self._mark_settled(self._history[-(len(calls) + 1) :])\n                    await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                    raise\n\n                batch_results: list[ToolResult] = []\n                for index, call in enumerate(calls):\n                    result = results[index]\n                    batch_results.append(result)\n                    run_tool_results.append(result)\n                    self._append_unsettled(\n                        ToolResultMessage(call.id, call.name, result)\n                    )\n                self._mark_settled(self._history[-(len(calls) + 1) :])\n                tool_calls += len(calls)\n                await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                if self._settled_turn_handler is not None:\n                    await self._settled_turn_handler()\n                if batch_results and all(result.terminate for result in batch_results):\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n                reached = guard_status()\n                if reached is not None:\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        StopReason.ABORTED,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                        status=reached,\n                    )\n                if turn_input is not None and turn_input.pending():\n                    steering = tuple(turn_input.take())\n                    to_model_messages(steering)\n                    self._append_settled(steering)\n        except asyncio.CancelledError:\n            message = AgentMessage.text(Role.ASSISTANT, "".join(text_parts))\n            outcome = AssistantOutcome(\n                message,\n                StopReason.ABORTED,\n                _add_usage(run_usage, current_usage),\n                attempts=max(total_attempts, 1),\n                tool_results=tuple(run_tool_results),\n                status=cancellation.status,\n            )\n            self._append_settled((message,))\n            await emit(\n                EventType.RUN_CANCELLED,\n                attempt=max(total_attempts, 1),\n                partial_text="".join(text_parts),\n                partial_usage=current_usage,\n            )\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n        finally:\n            await events.put(_EVENTS_DONE)\n'


In [ ]:
SESSION_SOURCE = '"""Application-facing control for one in-memory agent conversation."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections import deque\nfrom dataclasses import dataclass\nfrom collections.abc import AsyncIterator, Sequence\nfrom enum import Enum\nfrom typing import cast\n\nfrom .compaction import (\n    CharacterTokenEstimator,\n    CompactionCheckpoint,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarning,\n    CompactionWarningCode,\n    StructuredSummary,\n    TokenEstimator,\n)\nfrom .model import AgentMessage, Role, StopReason\nfrom .persistence import SessionBusyError, SessionStore, SessionWriter\nfrom .runtime import AgentRunHandle, AgentRuntime, AssistantOutcome, RuntimeEvent\n\n\n_SESSION_EVENTS_DONE = object()\n\n\nclass InputKind(str, Enum):\n    STEERING = "steering"\n    FOLLOW_UP = "follow_up"\n\n\n@dataclass(frozen=True, slots=True)\nclass PendingInput:\n    kind: InputKind\n    message: AgentMessage\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionRunResult:\n    outcome: AssistantOutcome\n    outcomes: tuple[AssistantOutcome, ...] = ()\n    pending_inputs: tuple[PendingInput, ...] = ()\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionResult:\n    checkpoint: CompactionCheckpoint\n\n\nclass SessionRunHandle:\n    def __init__(\n        self,\n        task: asyncio.Task[SessionRunResult],\n        session: "AgentSession",\n        events: asyncio.Queue[RuntimeEvent | object],\n    ) -> None:\n        self._task = task\n        self._session = session\n        self._events = events\n\n    async def result(self) -> SessionRunResult:\n        return await self._task\n\n    def cancel(self) -> None:\n        if not self._task.done():\n            self._session.cancel()\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _SESSION_EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n\nclass _SteeringQueue:\n    def __init__(self) -> None:\n        self._messages: deque[PendingInput] = deque()\n\n    def append(self, message: AgentMessage) -> None:\n        self._messages.append(PendingInput(InputKind.STEERING, message))\n\n    def pending(self) -> bool:\n        return bool(self._messages)\n\n    def take(self) -> Sequence[AgentMessage]:\n        return (self._messages.popleft().message,)\n\n    def drain(self) -> tuple[PendingInput, ...]:\n        drained = tuple(self._messages)\n        self._messages.clear()\n        return drained\n\n\nclass AgentSession:\n    """Coordinate one active Run with optional durable Session state."""\n\n    def __init__(\n        self,\n        runtime: AgentRuntime,\n        *,\n        store: SessionStore | None = None,\n        session_id: str | None = None,\n        parent_entry_id: str | None = None,\n        compaction_policy: CompactionPolicy | None = None,\n        compaction_strategy: CompactionStrategy | None = None,\n        token_estimator: TokenEstimator | None = None,\n    ) -> None:\n        if not isinstance(runtime, AgentRuntime):\n            raise TypeError("runtime must be an AgentRuntime")\n        if (store is None) != (session_id is None):\n            raise ValueError("durable Sessions require both store and session_id")\n        if compaction_policy is not None and not isinstance(\n            compaction_policy, CompactionPolicy\n        ):\n            raise TypeError("compaction_policy must be a CompactionPolicy")\n        if compaction_strategy is not None and not callable(\n            getattr(compaction_strategy, "plan", None)\n        ):\n            raise TypeError("compaction_strategy must provide plan()")\n        if token_estimator is not None and not callable(\n            getattr(token_estimator, "estimate", None)\n        ):\n            raise TypeError("token_estimator must provide estimate()")\n        self._runtime = runtime\n        self._store = store\n        self._session_id = session_id\n        self._parent_entry_id = parent_entry_id\n        self._compaction_policy = compaction_policy or CompactionPolicy()\n        self._compaction_strategy = compaction_strategy or CompactionStrategy()\n        self._token_estimator = token_estimator or CharacterTokenEstimator()\n        self._warnings: list[CompactionWarning] = []\n        self._busy = False\n        self._active: AgentRunHandle | None = None\n        self._compaction_task: asyncio.Task[object] | None = None\n        self._cancel_requested = False\n        self._steering = _SteeringQueue()\n        self._follow_ups: deque[PendingInput] = deque()\n\n    @property\n    def busy(self) -> bool:\n        return self._busy\n\n    @property\n    def session_id(self) -> str | None:\n        return self._session_id\n\n    @property\n    def warnings(self) -> tuple[CompactionWarning, ...]:\n        return tuple(self._warnings)\n\n    def start(self, prompt: str | AgentMessage) -> SessionRunHandle:\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Session already has an active Run",\n            )\n        message = (\n            AgentMessage.text(Role.USER, prompt) if isinstance(prompt, str) else prompt\n        )\n        if not isinstance(message, AgentMessage) or message.role is not Role.USER:\n            raise TypeError("prompt must be text or a user AgentMessage")\n        writer: SessionWriter | None = None\n        if self._store is not None:\n            assert self._session_id is not None\n            writer = self._store.writer(self._session_id)\n            writer = writer.__enter__()\n        self._busy = True\n        self._cancel_requested = False\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        try:\n            task = asyncio.get_running_loop().create_task(\n                self._drive(message, events, writer)\n            )\n        except BaseException:\n            self._busy = False\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            raise\n        return SessionRunHandle(task, self, events)\n\n    async def run(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.start(prompt).result()\n\n    async def prompt(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.run(prompt)\n\n    async def compact(self, focus: str | None = None) -> CompactionResult:\n        """Create and install a context checkpoint at a Settled Boundary."""\n\n        if focus is not None and not focus.strip():\n            raise ValueError("Compaction focus cannot be empty")\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Compaction requires an idle Session at a Settled Boundary",\n            )\n        writer: SessionWriter | None = None\n        if self._store is not None:\n            assert self._session_id is not None\n            writer = self._store.writer(self._session_id).__enter__()\n        self._busy = True\n        try:\n            return await self._compact(\n                CompactionTrigger.MANUAL,\n                focus=focus,\n                writer=writer,\n            )\n        finally:\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            self._busy = False\n\n    async def _compact(\n        self,\n        trigger: CompactionTrigger,\n        *,\n        focus: str | None,\n        writer: SessionWriter | None,\n    ) -> CompactionResult:\n        current = asyncio.current_task()\n        previous = self._compaction_task\n        self._compaction_task = cast(asyncio.Task[object] | None, current)\n        try:\n            policy = self._compaction_policy.resolve(self._runtime.model)\n            plan = self._compaction_strategy.plan(\n                self._runtime.effective_history,\n                keep_recent_tokens=policy.keep_recent_tokens,\n                estimator=self._token_estimator,\n            )\n            generated = await self._runtime.generate_summary(plan.source, focus=focus)\n            checkpoint = CompactionCheckpoint(\n                trigger=trigger,\n                summary=StructuredSummary(generated.text, focus=focus),\n                tokens_before=plan.tokens_before,\n                summary_usage=generated.usage,\n                retained_tail=plan.retained_tail,\n            )\n            if writer is not None:\n                entry = writer.append_compaction(\n                    checkpoint,\n                    parent_id=self._parent_entry_id,\n                )\n                self._parent_entry_id = entry.entry_id\n            self._runtime.install_compaction(checkpoint)\n            return CompactionResult(checkpoint)\n        finally:\n            self._compaction_task = previous\n\n    async def _maybe_compact_after_settlement(\n        self,\n        writer: SessionWriter | None,\n    ) -> CompactionResult | None:\n        resolved = self._compaction_policy.resolve(self._runtime.model)\n        threshold = resolved.threshold_tokens\n        if threshold is None:\n            if (\n                resolved.context_window is None\n                and not any(\n                    warning.code is CompactionWarningCode.CONTEXT_WINDOW_UNKNOWN\n                    for warning in self._warnings\n                )\n            ):\n                self._warnings.append(\n                    CompactionWarning(\n                        CompactionWarningCode.CONTEXT_WINDOW_UNKNOWN,\n                        "threshold Compaction disabled because ModelSpec has no "\n                        "context_window; normalized overflow recovery remains enabled",\n                    )\n                )\n            return None\n        current_tokens = self._token_estimator.estimate(\n            self._runtime.effective_history\n        )\n        if current_tokens <= threshold:\n            return None\n        try:\n            return await self._compact(\n                CompactionTrigger.THRESHOLD,\n                focus=None,\n                writer=writer,\n            )\n        except Exception as error:\n            self._warnings.append(\n                CompactionWarning(\n                    CompactionWarningCode.THRESHOLD_FAILED,\n                    "threshold Compaction failed; Session history unchanged: "\n                    f"{type(error).__name__}",\n                )\n            )\n            return None\n\n    def steer(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Steering requires an active Run")\n        self._steering.append(self._user_message(message))\n\n    def follow_up(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Follow-up requires an active Run")\n        self._follow_ups.append(\n            PendingInput(InputKind.FOLLOW_UP, self._user_message(message))\n        )\n\n    def cancel(self) -> None:\n        self._cancel_requested = True\n        if self._compaction_task is not None and not self._compaction_task.done():\n            self._compaction_task.cancel()\n        if self._active is not None:\n            self._active.cancel()\n\n    async def _drive(\n        self,\n        message: AgentMessage,\n        events: asyncio.Queue[RuntimeEvent | object],\n        writer: SessionWriter | None,\n    ) -> SessionRunResult:\n        outcomes: list[AssistantOutcome] = []\n\n        if writer is not None:\n            def settle(messages) -> None:\n                entry = writer.append(messages, parent_id=self._parent_entry_id)\n                self._parent_entry_id = entry.entry_id\n\n            self._runtime.set_settlement_sink(settle)\n\n        async def recover_context_overflow() -> bool:\n            await self._compact(\n                CompactionTrigger.OVERFLOW,\n                focus=None,\n                writer=writer,\n            )\n            return True\n\n        self._runtime.set_context_overflow_recovery(recover_context_overflow)\n        self._runtime.set_settled_turn_handler(\n            lambda: self._maybe_compact_after_settlement(writer)\n        )\n\n        async def forward(handle: AgentRunHandle) -> None:\n            async for event in handle.events():\n                await events.put(event)\n\n        try:\n            next_message = message\n            while True:\n                self._active = self._runtime.start(\n                    [next_message], turn_input=self._steering\n                )\n                if self._cancel_requested:\n                    self._active.cancel()\n                forwarding = asyncio.create_task(forward(self._active))\n                outcome = await self._active.result()\n                await forwarding\n                outcomes.append(outcome)\n                if outcome.stop_reason in {StopReason.ABORTED, StopReason.ERROR}:\n                    break\n                if not self._follow_ups:\n                    break\n                next_message = self._follow_ups.popleft().message\n            pending = self._steering.drain() + tuple(self._follow_ups)\n            self._follow_ups.clear()\n            return SessionRunResult(outcome, tuple(outcomes), pending)\n        finally:\n            self._runtime.set_settlement_sink(None)\n            self._runtime.set_context_overflow_recovery(None)\n            self._runtime.set_settled_turn_handler(None)\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            self._active = None\n            self._cancel_requested = False\n            self._busy = False\n            await events.put(_SESSION_EVENTS_DONE)\n\n    @staticmethod\n    def _user_message(message: str | AgentMessage) -> AgentMessage:\n        accepted = (\n            AgentMessage.text(Role.USER, message)\n            if isinstance(message, str)\n            else message\n        )\n        if not isinstance(accepted, AgentMessage) or accepted.role is not Role.USER:\n            raise TypeError("Session input must be text or a user AgentMessage")\n        return accepted\n\n\ndef create_agent_session(\n    runtime: AgentRuntime,\n    *,\n    store: SessionStore | None = None,\n    session_id: str | None = None,\n    fork_from: str | None = None,\n    no_save: bool = False,\n    compaction_policy: CompactionPolicy | None = None,\n    compaction_strategy: CompactionStrategy | None = None,\n    token_estimator: TokenEstimator | None = None,\n) -> AgentSession:\n    """Create a new durable Session, explicitly continue/fork one, or opt out."""\n\n    if no_save:\n        if store is not None or session_id is not None or fork_from is not None:\n            raise ValueError("no_save cannot be combined with persistence or continuation")\n        return AgentSession(\n            runtime,\n            compaction_policy=compaction_policy,\n            compaction_strategy=compaction_strategy,\n            token_estimator=token_estimator,\n        )\n    if store is None:\n        if session_id is not None or fork_from is not None:\n            raise ValueError("continuation requires a SessionStore")\n        return AgentSession(\n            runtime,\n            compaction_policy=compaction_policy,\n            compaction_strategy=compaction_strategy,\n            token_estimator=token_estimator,\n        )\n    if runtime.history:\n        raise ValueError("a durable AgentSession requires a fresh AgentRuntime")\n    if session_id is None:\n        if fork_from is not None:\n            raise ValueError("fork_from requires an existing session_id")\n        state = store.create()\n        leaf = None\n    else:\n        state = store.read(session_id)\n        leaf = fork_from if fork_from is not None else state.active_leaf_id\n        runtime.restore_history(\n            state.history(leaf),\n            effective_history=state.effective_history(leaf),\n        )\n    return AgentSession(\n        runtime,\n        store=store,\n        session_id=state.session_id,\n        parent_entry_id=leaf,\n        compaction_policy=compaction_policy,\n        compaction_strategy=compaction_strategy,\n        token_estimator=token_estimator,\n    )\n'


In [ ]:
INIT_SOURCE = 'from .compaction import (\n    CharacterTokenEstimator,\n    CompactionCheckpoint,\n    CompactionPlan,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarning,\n    CompactionWarningCode,\n    ResolvedCompactionPolicy,\n    StructuredSummary,\n    TokenEstimator,\n)\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelMessage,\n    ModelOperation,\n    ModelProtocolError,\n    ModelRequest,\n    ModelResult,\n    ModelSpec,\n    ModelToolResultMessage,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n    ScriptedModelAdapter,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    UnsupportedContentError,\n    Usage,\n    UsageUpdate,\n    complete,\n    to_model_messages,\n)\nfrom .persistence import (\n    ConversationMessage,\n    JSONLSessionStore,\n    JSONLSessionWriter,\n    MemorySessionStore,\n    MemorySessionWriter,\n    MigrationResult,\n    RecoveryCode,\n    RecoveryWarning,\n    SESSION_SCHEMA_VERSION,\n    SchemaVersion,\n    SessionBusyError,\n    SessionEntry,\n    SessionStore,\n    SessionState,\n    SessionWriter,\n    UnsupportedSchemaVersionError,\n    migrate_session_file,\n)\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    EventType,\n    RetryPolicy,\n    RunGuard,\n    RuntimeEvent,\n    Sleeper,\n    ContextOverflowRecovery,\n    SettledTurnHandler,\n    SummaryGeneration,\n    TerminalStatus,\n    TurnInput,\n)\nfrom .session import (\n    AgentSession,\n    CompactionResult,\n    InputKind,\n    PendingInput,\n    SessionRunHandle,\n    SessionRunResult,\n    create_agent_session,\n)\nfrom .tools import (\n    AsyncioProcessOperations,\n    CompleteOutputKind,\n    CompleteOutputReference,\n    LocalToolExecutor,\n    PreparedToolCall,\n    ProcessResult,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\n\n__all__ = [name for name in globals() if not name.startswith("_")]\n'


In [ ]:
PY_TYPED_SOURCE = ''


In [ ]:
import asyncio
import importlib
import shutil
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix='chapter-06-minimal-') as temporary:
    package = Path(temporary) / 'agent_harness'
    shutil.copytree(CHAPTER_5 / 'src' / 'agent_harness', package)
    for name, text in {
        'compaction.py': COMPACTION_SOURCE, 'model.py': MODEL_SOURCE,
        'persistence.py': PERSISTENCE_SOURCE, 'runtime.py': RUNTIME_SOURCE,
        'session.py': SESSION_SOURCE, '__init__.py': INIT_SOURCE,
    }.items():
        (package / name).write_text(text, encoding='utf-8')
    sys.path.insert(0, temporary)
    chapter6 = importlib.import_module('agent_harness')

    async def minimal_compaction():
        adapter = chapter6.ScriptedModelAdapter([
            chapter6.TextDelta('Keep the migration decision.'),
            chapter6.UsageUpdate(chapter6.Usage(8, 6, 14)),
            chapter6.ModelEnd(chapter6.StopReason.COMPLETE),
        ])
        runtime = chapter6.AgentRuntime(
            adapter, chapter6.ModelSpec('scripted/ch06'),
            history=(
                chapter6.AgentMessage.text(chapter6.Role.USER, 'old question'),
                chapter6.AgentMessage.text(chapter6.Role.ASSISTANT, 'recent answer'),
            ),
        )
        session = chapter6.AgentSession(
            runtime,
            compaction_policy=chapter6.CompactionPolicy(keep_recent_tokens=8),
        )
        return adapter, runtime, await session.compact('preserve decisions')

    minimal_adapter, minimal_runtime, minimal_result = await minimal_compaction()
    assert minimal_result.checkpoint.summary_usage == chapter6.Usage(8, 6, 14)
    assert minimal_adapter.received_requests[0].operation is chapter6.ModelOperation.COMPACTION
    assert minimal_runtime.history[0].content[0].text == 'old question'
    sys.path.remove(temporary)
    for module_name in tuple(sys.modules):
        if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
            del sys.modules[module_name]


## Staged Construction

The tracer bullet first persisted one manual checkpoint and continued through its effective context. Later red–green slices added adaptive threshold formulas, Tool batch cut points, durable JSONL reconstruction, retryable summary generation, cancellation, warnings, and the one-attempt overflow recovery path.

In [ ]:
tool_call = chapter6.AgentMessage(
    chapter6.Role.ASSISTANT,
    (chapter6.ToolCallContent('call-1', 'lookup', '{}'),),
)
tool_result = chapter6.ToolResultMessage(
    'call-1', 'lookup', chapter6.ToolResult('large result')
)
structural_plan = chapter6.CompactionStrategy().plan(
    (chapter6.AgentMessage.text(chapter6.Role.USER, 'prefix'), tool_call, tool_result),
    keep_recent_tokens=1,
    estimator=chapter6.CharacterTokenEstimator(),
)
assert structural_plan.retained_tail == (tool_call, tool_result)


## Observable Trace

Every ModelRequest identifies `run` or `compaction`. Overflow recovery also emits ordered `compaction_start` and `compaction_end` Runtime Events. These are diagnostic observations; the durable checkpoint remains the resumable state.

In [ ]:
class OverflowOnceAdapter:
    def __init__(self):
        self.requests = []
        self.run_attempts = 0

    async def stream(self, request):
        self.requests.append(request)
        if request.operation is chapter6.ModelOperation.COMPACTION:
            yield chapter6.TextDelta('overflow summary')
            yield chapter6.ModelEnd(chapter6.StopReason.COMPLETE)
            return
        self.run_attempts += 1
        if self.run_attempts == 1:
            raise chapter6.ModelAdapterError(chapter6.ModelError(
                chapter6.ModelErrorCode.CONTEXT_OVERFLOW, 'too large', False
            ))
        yield chapter6.TextDelta('recovered')
        yield chapter6.ModelEnd(chapter6.StopReason.COMPLETE)

async def overflow_demo():
    adapter = OverflowOnceAdapter()
    runtime = chapter6.AgentRuntime(
        adapter, chapter6.ModelSpec('scripted/overflow'),
        history=(
            chapter6.AgentMessage.text(chapter6.Role.USER, 'old'),
            chapter6.AgentMessage.text(chapter6.Role.ASSISTANT, 'answer'),
        ),
    )
    session = chapter6.AgentSession(
        runtime, compaction_policy=chapter6.CompactionPolicy(keep_recent_tokens=1)
    )
    handle = session.start('continue')
    events_task = asyncio.create_task(
        _collect_events(handle)
    )
    result = await handle.result()
    return adapter, result, await events_task

async def _collect_events(handle):
    return [event async for event in handle.events()]

overflow_adapter, overflow_result, overflow_events = await overflow_demo()
assert overflow_result.outcome.message.content[0].text == 'recovered'
assert [request.operation.value for request in overflow_adapter.requests] == [
    'run', 'compaction', 'run'
]
assert {'compaction_start', 'compaction_end'} <= {
    event.type.value for event in overflow_events
}


## Failure Boundaries and Trade-offs

Threshold Compaction runs only after settled work. Its failure records a warning and leaves history unchanged. A context overflow can invoke one Compaction and one retry; a failed recovery or second overflow terminates visibly. Summary cancellation commits nothing. The default estimator is deterministic and offline, not a substitute for Provider-reported usage.

In [ ]:
small = chapter6.CompactionPolicy().resolve(
    chapter6.ModelSpec('small', context_window=32768)
)
broad = chapter6.CompactionPolicy().resolve(
    chapter6.ModelSpec('broad', context_window=131072)
)
assert (small.reserve_tokens, small.keep_recent_tokens) == (4096, 8192)
assert (broad.reserve_tokens, broad.keep_recent_tokens) == (16384, 20000)


## Checkpoint Export and Verification

Chapter metadata names Chapter 5 as the base Checkpoint. Export carries forward unchanged modules and all prior regression tests, replaces the evolved files, adds Chapter 6 tests, writes a deterministic manifest, then compiles, installs offline, imports, and runs the cumulative suite before publishing.

In [ ]:
COMPACTION_TEST_SOURCE = 'from __future__ import annotations\n\nimport asyncio\n\nimport pytest\n\nfrom agent_harness import (\n    AgentMessage,\n    AgentRuntime,\n    AgentSession,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarningCode,\n    EventType,\n    JSONLSessionStore,\n    MemorySessionStore,\n    ModelEnd,\n    ModelAdapterError,\n    ModelError,\n    ModelErrorCode,\n    ModelOperation,\n    ModelSpec,\n    Role,\n    ScriptedModelAdapter,\n    StopReason,\n    StructuredSummary,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    Tool,\n    ToolResult,\n    ToolResultMessage,\n    Usage,\n    UsageUpdate,\n    create_agent_session,\n    migrate_session_file,\n)\n\n\nclass WordTokenEstimator:\n    def estimate(self, messages) -> int:\n        return sum(\n            len(block.text.split())\n            for message in messages\n            for block in getattr(message, "content", ())\n            if hasattr(block, "text")\n        )\n\n\ndef test_manual_compaction_persists_a_checkpoint_and_changes_only_model_context() -> None:\n    async def scenario() -> None:\n        original = (\n            AgentMessage.text(Role.USER, "old question with details"),\n            AgentMessage.text(Role.ASSISTANT, "old answer with rationale"),\n            AgentMessage.text(Role.USER, "recent question"),\n            AgentMessage.text(Role.ASSISTANT, "recent answer"),\n        )\n        store = MemorySessionStore()\n        state = store.create("manual-compaction")\n        with store.writer(state.session_id) as writer:\n            writer.append(original)\n        adapter = ScriptedModelAdapter(\n            [\n                [\n                    TextDelta("Decided to retain the safe migration path."),\n                    UsageUpdate(Usage(9, 8, 17)),\n                    ModelEnd(StopReason.COMPLETE),\n                ],\n                [TextDelta("continued from checkpoint"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        session = create_agent_session(\n            AgentRuntime(adapter, ModelSpec("scripted/compact")),\n            store=store,\n            session_id=state.session_id,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=2),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        compacted = await session.compact("preserve migration decisions")\n\n        assert compacted.checkpoint.trigger is CompactionTrigger.MANUAL\n        assert compacted.checkpoint.summary == StructuredSummary(\n            "Decided to retain the safe migration path.",\n            focus="preserve migration decisions",\n        )\n        assert compacted.checkpoint.tokens_before == 12\n        assert compacted.checkpoint.summary_usage == Usage(9, 8, 17)\n        assert compacted.checkpoint.retained_tail == (\n            AgentMessage.text(Role.ASSISTANT, "recent answer"),\n        )\n        recovered = store.read(state.session_id)\n        assert recovered.history() == original\n        assert recovered.compactions() == (compacted.checkpoint,)\n\n        await session.run("what next?")\n\n        assert adapter.received_requests[0].operation is ModelOperation.COMPACTION\n        continuation = adapter.received_requests[1]\n        assert continuation.operation is ModelOperation.RUN\n        assert continuation.messages == (\n            *compacted.checkpoint.model_context(),\n            AgentMessage.text(Role.USER, "what next?").to_model(),\n        )\n        assert store.read(state.session_id).history() == (\n            *original,\n            AgentMessage.text(Role.USER, "what next?"),\n            AgentMessage.text(Role.ASSISTANT, "continued from checkpoint"),\n        )\n\n    asyncio.run(scenario())\n\n\n@pytest.mark.parametrize(\n    ("model", "reserve", "keep", "threshold"),\n    [\n        (ModelSpec("small", context_window=32_768), 4_096, 8_192, 28_672),\n        (ModelSpec("broad", context_window=131_072), 16_384, 20_000, 114_688),\n        (ModelSpec("unknown"), None, 20_000, None),\n    ],\n)\ndef test_compaction_policy_resolves_adaptive_small_and_broad_context_defaults(\n    model: ModelSpec,\n    reserve: int | None,\n    keep: int,\n    threshold: int | None,\n) -> None:\n    resolved = CompactionPolicy().resolve(model)\n\n    assert resolved.reserve_tokens == reserve\n    assert resolved.keep_recent_tokens == keep\n    assert resolved.threshold_tokens == threshold\n\n\ndef test_retained_tail_keeps_an_oversized_tool_batch_structurally_complete() -> None:\n    class UnitEstimator:\n        def estimate(self, messages) -> int:\n            return len(messages)\n\n    tool_call = AgentMessage(\n        Role.ASSISTANT,\n        (ToolCallContent("call-1", "lookup", \'{}\'),),\n    )\n    tool_result = ToolResultMessage("call-1", "lookup", ToolResult("large result"))\n\n    plan = CompactionStrategy().plan(\n        (\n            AgentMessage.text(Role.USER, "summarize this prefix"),\n            tool_call,\n            tool_result,\n        ),\n        keep_recent_tokens=1,\n        estimator=UnitEstimator(),\n    )\n\n    assert plan.source == (AgentMessage.text(Role.USER, "summarize this prefix"),)\n    assert plan.retained_tail == (tool_call, tool_result)\n\n\ndef test_threshold_compaction_runs_after_the_turn_settles() -> None:\n    async def scenario() -> None:\n        adapter = ScriptedModelAdapter(\n            [\n                [TextDelta("settled answer"), ModelEnd(StopReason.COMPLETE)],\n                [TextDelta("threshold summary"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        store = MemorySessionStore()\n        session = create_agent_session(\n            AgentRuntime(\n                adapter,\n                ModelSpec(\n                    "scripted/threshold",\n                    context_window=20,\n                    max_output_tokens=4,\n                ),\n            ),\n            store=store,\n            compaction_policy=CompactionPolicy(),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        result = await session.run(" ".join(f"word-{index}" for index in range(20)))\n\n        assert result.outcome.stop_reason is StopReason.COMPLETE\n        assert store.read(session.session_id).compactions()[0].trigger is (\n            CompactionTrigger.THRESHOLD\n        )\n        assert [request.operation for request in adapter.received_requests] == [\n            ModelOperation.RUN,\n            ModelOperation.COMPACTION,\n        ]\n\n    asyncio.run(scenario())\n\n\ndef test_failed_threshold_compaction_warns_without_changing_settled_history() -> None:\n    class ThresholdFailureAdapter:\n        def __init__(self) -> None:\n            self.requests = []\n\n        async def stream(self, request):\n            self.requests.append(request)\n            if request.operation is ModelOperation.COMPACTION:\n                raise ModelAdapterError(\n                    ModelError(\n                        ModelErrorCode.PROVIDER,\n                        "summary unavailable",\n                        retryable=False,\n                    )\n                )\n            yield TextDelta("settled answer")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = ThresholdFailureAdapter()\n        store = MemorySessionStore()\n        session = create_agent_session(\n            AgentRuntime(\n                adapter,\n                ModelSpec(\n                    "scripted/threshold-failure",\n                    context_window=20,\n                    max_output_tokens=4,\n                ),\n            ),\n            store=store,\n            token_estimator=WordTokenEstimator(),\n        )\n        prompt = " ".join(f"word-{index}" for index in range(20))\n\n        result = await session.run(prompt)\n\n        assert result.outcome.stop_reason is StopReason.COMPLETE\n        state = store.read(session.session_id)\n        assert state.history() == (\n            AgentMessage.text(Role.USER, prompt),\n            AgentMessage.text(Role.ASSISTANT, "settled answer"),\n        )\n        assert state.compactions() == ()\n        assert session.warnings[-1].code is CompactionWarningCode.THRESHOLD_FAILED\n        assert "history unchanged" in session.warnings[-1].message\n\n    asyncio.run(scenario())\n\n\ndef test_context_overflow_compacts_once_and_retries_outside_transient_backoff() -> None:\n    class OverflowThenSuccessAdapter:\n        def __init__(self) -> None:\n            self.requests = []\n            self.run_attempts = 0\n\n        async def stream(self, request):\n            self.requests.append(request)\n            if request.operation is ModelOperation.COMPACTION:\n                yield TextDelta("overflow recovery summary")\n                yield ModelEnd(StopReason.COMPLETE)\n                return\n            self.run_attempts += 1\n            if self.run_attempts == 1:\n                raise ModelAdapterError(\n                    ModelError(\n                        ModelErrorCode.CONTEXT_OVERFLOW,\n                        "context capacity exceeded",\n                        retryable=False,\n                    )\n                )\n            yield TextDelta("recovered answer")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = OverflowThenSuccessAdapter()\n        sleeps: list[float] = []\n\n        async def sleeper(delay: float) -> None:\n            sleeps.append(delay)\n\n        store = MemorySessionStore()\n        state = store.create("overflow-recovery")\n        with store.writer(state.session_id) as writer:\n            writer.append(\n                (\n                    AgentMessage.text(Role.USER, "old question"),\n                    AgentMessage.text(Role.ASSISTANT, "old answer"),\n                )\n            )\n        session = create_agent_session(\n            AgentRuntime(\n                adapter,\n                ModelSpec("scripted/overflow"),\n                sleeper=sleeper,\n            ),\n            store=store,\n            session_id=state.session_id,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        handle = session.start("continue")\n        events_task = asyncio.create_task(_collect_events(handle))\n        result = await handle.result()\n        events = await events_task\n\n        assert result.outcome.message == AgentMessage.text(\n            Role.ASSISTANT, "recovered answer"\n        )\n        assert result.outcome.attempts == 2\n        assert sleeps == []\n        assert [request.operation for request in adapter.requests] == [\n            ModelOperation.RUN,\n            ModelOperation.COMPACTION,\n            ModelOperation.RUN,\n        ]\n        assert [checkpoint.trigger for checkpoint in store.read(state.session_id).compactions()] == [\n            CompactionTrigger.OVERFLOW\n        ]\n        assert EventType.COMPACTION_START in [event.type for event in events]\n        assert EventType.COMPACTION_END in [event.type for event in events]\n\n    async def _collect_events(handle):\n        return [event async for event in handle.events()]\n\n    asyncio.run(scenario())\n\n\ndef test_failed_overflow_compaction_terminates_the_run_clearly() -> None:\n    class FailedRecoveryAdapter:\n        def __init__(self) -> None:\n            self.requests = []\n\n        async def stream(self, request):\n            self.requests.append(request)\n            if request.operation is ModelOperation.COMPACTION:\n                raise ModelAdapterError(\n                    ModelError(\n                        ModelErrorCode.PROVIDER,\n                        "summary unavailable",\n                        retryable=False,\n                    )\n                )\n            raise ModelAdapterError(\n                ModelError(\n                    ModelErrorCode.CONTEXT_OVERFLOW,\n                    "context capacity exceeded",\n                    retryable=False,\n                )\n            )\n            yield  # pragma: no cover - keeps this an async generator\n\n    async def scenario() -> None:\n        adapter = FailedRecoveryAdapter()\n        store = MemorySessionStore()\n        state = store.create("failed-overflow-recovery")\n        with store.writer(state.session_id) as writer:\n            writer.append(\n                (\n                    AgentMessage.text(Role.USER, "old question"),\n                    AgentMessage.text(Role.ASSISTANT, "old answer"),\n                )\n            )\n        session = create_agent_session(\n            AgentRuntime(adapter, ModelSpec("scripted/failed-overflow")),\n            store=store,\n            session_id=state.session_id,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        result = await session.run("continue")\n\n        assert result.outcome.stop_reason is StopReason.ERROR\n        assert result.outcome.error is not None\n        assert result.outcome.error.code is ModelErrorCode.COMPACTION_FAILED\n        assert "Compaction failed" in result.outcome.error.message\n        assert [request.operation for request in adapter.requests] == [\n            ModelOperation.RUN,\n            ModelOperation.COMPACTION,\n        ]\n        assert store.read(state.session_id).compactions() == ()\n\n    asyncio.run(scenario())\n\n\ndef test_manual_compaction_uses_session_cancellation_and_commits_nothing() -> None:\n    class BlockingSummaryAdapter:\n        def __init__(self) -> None:\n            self.started = asyncio.Event()\n\n        async def stream(self, request):\n            assert request.operation is ModelOperation.COMPACTION\n            self.started.set()\n            await asyncio.Event().wait()\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = BlockingSummaryAdapter()\n        store = MemorySessionStore()\n        state = store.create("cancel-compaction")\n        original = (\n            AgentMessage.text(Role.USER, "old question"),\n            AgentMessage.text(Role.ASSISTANT, "old answer"),\n        )\n        with store.writer(state.session_id) as writer:\n            writer.append(original)\n        session = create_agent_session(\n            AgentRuntime(adapter, ModelSpec("scripted/cancel-compaction")),\n            store=store,\n            session_id=state.session_id,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            token_estimator=WordTokenEstimator(),\n        )\n        task = asyncio.create_task(session.compact())\n        await adapter.started.wait()\n\n        session.cancel()\n\n        with pytest.raises(asyncio.CancelledError):\n            await asyncio.wait_for(task, timeout=0.2)\n        assert session.busy is False\n        assert store.read(state.session_id).history() == original\n        assert store.read(state.session_id).compactions() == ()\n\n    asyncio.run(scenario())\n\n\ndef test_jsonl_compaction_checkpoint_reopens_with_materialized_effective_context(\n    tmp_path,\n) -> None:\n    async def scenario() -> None:\n        store = JSONLSessionStore(tmp_path)\n        state = store.create("durable-compaction")\n        original = (\n            AgentMessage.text(Role.USER, "old question"),\n            AgentMessage.text(Role.ASSISTANT, "recent answer"),\n        )\n        with store.writer(state.session_id) as writer:\n            writer.append(original)\n        session = create_agent_session(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("durable summary"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/durable-compaction"),\n            ),\n            store=store,\n            session_id=state.session_id,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=2),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        result = await session.compact()\n        reopened = JSONLSessionStore(tmp_path).read(state.session_id)\n\n        assert reopened.history() == original\n        assert reopened.compactions() == (result.checkpoint,)\n        assert reopened.effective_history() == (\n            result.checkpoint.summary.as_message(),\n            *result.checkpoint.retained_tail,\n        )\n\n        continuation_adapter = ScriptedModelAdapter(\n            [TextDelta("continued"), ModelEnd(StopReason.COMPLETE)]\n        )\n        continued = create_agent_session(\n            AgentRuntime(\n                continuation_adapter,\n                ModelSpec("scripted/reopen-compaction"),\n            ),\n            store=JSONLSessionStore(tmp_path),\n            session_id=state.session_id,\n        )\n        await continued.run("next")\n        assert continuation_adapter.received_requests[0].messages == (\n            *result.checkpoint.model_context(),\n            AgentMessage.text(Role.USER, "next").to_model(),\n        )\n\n        source = store.path_for(state.session_id)\n        original_bytes = source.read_bytes()\n        destination = tmp_path / "migrated" / "sessions" / source.name\n        migrate_session_file(source, destination)\n        migrated = JSONLSessionStore(tmp_path / "migrated").read(state.session_id)\n        assert source.read_bytes() == original_bytes\n        assert migrated.history() == JSONLSessionStore(tmp_path).read(\n            state.session_id\n        ).history()\n        assert migrated.compactions() == (result.checkpoint,)\n\n    asyncio.run(scenario())\n\n\ndef test_unknown_context_window_disables_threshold_compaction_with_one_warning() -> None:\n    async def scenario() -> None:\n        adapter = ScriptedModelAdapter(\n            [\n                [TextDelta("first"), ModelEnd(StopReason.COMPLETE)],\n                [TextDelta("second"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/unknown-window")),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        await session.run("a long first prompt")\n        await session.run("a long second prompt")\n\n        assert [warning.code for warning in session.warnings] == [\n            CompactionWarningCode.CONTEXT_WINDOW_UNKNOWN\n        ]\n        assert [request.operation for request in adapter.received_requests] == [\n            ModelOperation.RUN,\n            ModelOperation.RUN,\n        ]\n\n    asyncio.run(scenario())\n\n\ndef test_summary_generation_reuses_runtime_retry_policy_and_sleeper() -> None:\n    class RetrySummaryAdapter:\n        def __init__(self) -> None:\n            self.requests = []\n\n        async def stream(self, request):\n            self.requests.append(request)\n            if len(self.requests) == 1:\n                raise ModelAdapterError(\n                    ModelError(\n                        ModelErrorCode.RATE_LIMIT,\n                        "retry summary",\n                        retryable=True,\n                    )\n                )\n            yield TextDelta("summary after retry")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = RetrySummaryAdapter()\n        sleeps: list[float] = []\n\n        async def sleeper(delay: float) -> None:\n            sleeps.append(delay)\n\n        runtime = AgentRuntime(\n            adapter,\n            ModelSpec("scripted/retry-summary"),\n            sleeper=sleeper,\n        )\n        runtime.restore_history(\n            (\n                AgentMessage.text(Role.USER, "old question"),\n                AgentMessage.text(Role.ASSISTANT, "old answer"),\n            )\n        )\n        session = AgentSession(\n            runtime,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        result = await session.compact()\n\n        assert result.checkpoint.summary.text == "summary after retry"\n        assert sleeps == [2.0]\n        assert [request.operation for request in adapter.requests] == [\n            ModelOperation.COMPACTION,\n            ModelOperation.COMPACTION,\n        ]\n\n    asyncio.run(scenario())\n\n\ndef test_a_second_context_overflow_terminates_without_another_compaction() -> None:\n    class RepeatedOverflowAdapter:\n        def __init__(self) -> None:\n            self.requests = []\n\n        async def stream(self, request):\n            self.requests.append(request)\n            if request.operation is ModelOperation.COMPACTION:\n                yield TextDelta("one recovery summary")\n                yield ModelEnd(StopReason.COMPLETE)\n                return\n            raise ModelAdapterError(\n                ModelError(\n                    ModelErrorCode.CONTEXT_OVERFLOW,\n                    "still too large",\n                    retryable=False,\n                )\n            )\n            yield  # pragma: no cover - keeps this an async generator\n\n    async def scenario() -> None:\n        adapter = RepeatedOverflowAdapter()\n        runtime = AgentRuntime(adapter, ModelSpec("scripted/repeated-overflow"))\n        runtime.restore_history(\n            (\n                AgentMessage.text(Role.USER, "old question"),\n                AgentMessage.text(Role.ASSISTANT, "old answer"),\n            )\n        )\n        session = AgentSession(\n            runtime,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            token_estimator=WordTokenEstimator(),\n        )\n\n        result = await session.run("continue")\n\n        assert result.outcome.stop_reason is StopReason.ERROR\n        assert result.outcome.error is not None\n        assert result.outcome.error.code is ModelErrorCode.CONTEXT_OVERFLOW\n        assert [request.operation for request in adapter.requests] == [\n            ModelOperation.RUN,\n            ModelOperation.COMPACTION,\n            ModelOperation.RUN,\n        ]\n\n    asyncio.run(scenario())\n\n\ndef test_threshold_check_compacts_a_settled_tool_turn_before_model_continuation() -> None:\n    class WeightedEstimator:\n        def estimate(self, messages) -> int:\n            total = 0\n            for message in messages:\n                if isinstance(message, ToolResultMessage):\n                    total += 1\n                    continue\n                text = " ".join(\n                    block.text\n                    for block in message.content\n                    if hasattr(block, "text")\n                )\n                total += 10 if "large-prefix" in text else 1\n            return total\n\n    class ToolThenAnswerAdapter:\n        def __init__(self) -> None:\n            self.requests = []\n            self.run_turn = 0\n\n        async def stream(self, request):\n            self.requests.append(request)\n            if request.operation is ModelOperation.COMPACTION:\n                yield TextDelta("short summary")\n                yield ModelEnd(StopReason.COMPLETE)\n                return\n            self.run_turn += 1\n            if self.run_turn == 1:\n                yield ToolCallDelta(0, "call-1", "lookup", \'{}\')\n                yield ModelEnd(StopReason.TOOL_USE)\n                return\n            yield TextDelta("final answer")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        async def lookup(arguments: dict[str, object]) -> ToolResult:\n            return ToolResult("result")\n\n        adapter = ToolThenAnswerAdapter()\n        session = AgentSession(\n            AgentRuntime(\n                adapter,\n                ModelSpec(\n                    "scripted/threshold-tool-turn",\n                    context_window=6,\n                    max_output_tokens=1,\n                ),\n                tools=[Tool("lookup", "lookup", {"type": "object"}, lookup)],\n            ),\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            token_estimator=WeightedEstimator(),\n        )\n\n        result = await session.run("large-prefix")\n\n        assert result.outcome.message == AgentMessage.text(\n            Role.ASSISTANT, "final answer"\n        )\n        assert [request.operation for request in adapter.requests] == [\n            ModelOperation.RUN,\n            ModelOperation.COMPACTION,\n            ModelOperation.RUN,\n        ]\n        assert "short summary" in adapter.requests[-1].messages[0].content[0].text\n\n    asyncio.run(scenario())\n'


In [ ]:
OPENAI_TEST_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nfrom contextlib import contextmanager\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nimport json\nfrom threading import Thread\nfrom typing import Iterator\n\nimport pytest\n\nfrom agent_harness import (\n    AgentMessage,\n    ModelAdapterError,\n    ModelErrorCode,\n    ModelSpec,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n    StopReason,\n    TextContent,\n    ToolCallContent,\n    Usage,\n    complete,\n)\n\n\ndef _chunk(delta, *, finish_reason=None):\n    return {\n        \'id\': \'chatcmpl-local\',\n        \'object\': \'chat.completion.chunk\',\n        \'created\': 1,\n        \'model\': \'local-test\',\n        \'choices\': [\n            {\'index\': 0, \'delta\': delta, \'finish_reason\': finish_reason}\n        ],\n    }\n\n\n@contextmanager\ndef fake_openai_server() -> Iterator[tuple[str, list[dict[str, object]]]]:\n    requests: list[dict[str, object]] = []\n\n    class Handler(BaseHTTPRequestHandler):\n        def do_POST(self):\n            length = int(self.headers.get(\'content-length\', \'0\'))\n            body = json.loads(self.rfile.read(length))\n            requests.append(\n                {\'path\': self.path, \'authorization\': self.headers.get(\'authorization\'), \'body\': body}\n            )\n            if body[\'model\'] in {\'reject-me\', \'retry-me\', \'overflow-me\'}:\n                error_code = (\n                    \'context_length_exceeded\'\n                    if body[\'model\'] == \'overflow-me\'\n                    else None\n                )\n                payload = json.dumps(\n                    {\n                        \'error\': {\n                            \'message\': \'SECRET provider detail\',\n                            \'type\': \'invalid_request_error\',\n                            \'code\': error_code,\n                        }\n                    }\n                ).encode()\n                status = (\n                    429\n                    if body[\'model\'] == \'retry-me\'\n                    else 400\n                    if body[\'model\'] == \'overflow-me\'\n                    else 401\n                )\n                self.send_response(status)\n                self.send_header(\'content-type\', \'application/json\')\n                self.send_header(\'content-length\', str(len(payload)))\n                if status == 429:\n                    self.send_header(\'retry-after\', \'7\')\n                self.end_headers()\n                self.wfile.write(payload)\n                return\n\n            events = [\n                _chunk({\'role\': \'assistant\', \'content\': \'Hel\'}),\n                _chunk({\'content\': \'lo\'}),\n                _chunk(\n                    {\n                        \'tool_calls\': [\n                            {\n                                \'index\': 0,\n                                \'id\': \'call-1\',\n                                \'type\': \'function\',\n                                \'function\': {\'name\': \'read\', \'arguments\': \'{"path":\'},\n                            }\n                        ]\n                    }\n                ),\n                _chunk(\n                    {\'tool_calls\': [{\'index\': 0, \'function\': {\'arguments\': \'"README.md"}\'}}]}\n                ),\n                _chunk({}, finish_reason=\'tool_calls\'),\n                {\n                    \'id\': \'chatcmpl-local\',\n                    \'object\': \'chat.completion.chunk\',\n                    \'created\': 1,\n                    \'model\': \'local-test\',\n                    \'choices\': [],\n                    \'usage\': {\'prompt_tokens\': 4, \'completion_tokens\': 6, \'total_tokens\': 10},\n                },\n            ]\n            payload = \'\'.join(f\'data: {json.dumps(event)}\\n\\n\' for event in events)\n            payload += \'data: [DONE]\\n\\n\'\n            encoded = payload.encode()\n            self.send_response(200)\n            self.send_header(\'content-type\', \'text/event-stream\')\n            self.send_header(\'content-length\', str(len(encoded)))\n            self.end_headers()\n            self.wfile.write(encoded)\n\n        def log_message(self, format, *args):\n            return\n\n    server = ThreadingHTTPServer((\'127.0.0.1\', 0), Handler)\n    thread = Thread(target=server.serve_forever, daemon=True)\n    thread.start()\n    try:\n        host, port = server.server_address\n        yield f\'http://{host}:{port}/v1\', requests\n    finally:\n        server.shutdown()\n        server.server_close()\n        thread.join(timeout=5)\n\n\ndef test_openai_compatible_stream_is_translated_at_the_adapter_seam(monkeypatch):\n    monkeypatch.setenv(\'OPENAI_API_KEY\', \'ambient-key-must-not-win\')\n    with fake_openai_server() as (base_url, requests):\n        adapter = OpenAICompatibleAdapter(\n            OpenAICompatibleConfig(base_url=base_url, api_key=\'explicit-key\')\n        )\n        result = asyncio.run(\n            complete(\n                adapter,\n                [AgentMessage.text(Role.USER, \'inspect\')],\n                ModelSpec(\'local-test\', max_output_tokens=128),\n            )\n        )\n\n    assert result.message.content == (\n        TextContent(\'Hello\'),\n        ToolCallContent(\'call-1\', \'read\', \'{"path":"README.md"}\'),\n    )\n    assert result.stop_reason is StopReason.TOOL_USE\n    assert result.usage == Usage(4, 6, 10)\n    assert requests[0][\'path\'] == \'/v1/chat/completions\'\n    assert requests[0][\'authorization\'] == \'Bearer explicit-key\'\n    assert requests[0][\'body\'][\'messages\'] == [{\'role\': \'user\', \'content\': \'inspect\'}]\n\n\ndef test_openai_compatible_failure_is_normalized_without_raw_detail():\n    with fake_openai_server() as (base_url, _):\n        adapter = OpenAICompatibleAdapter(\n            OpenAICompatibleConfig(base_url=base_url, api_key=\'explicit-key\')\n        )\n        with pytest.raises(ModelAdapterError) as captured:\n            asyncio.run(\n                complete(\n                    adapter,\n                    [AgentMessage.text(Role.USER, \'fail safely\')],\n                    ModelSpec(\'reject-me\'),\n                )\n            )\n\n    assert captured.value.error.code is ModelErrorCode.AUTHENTICATION\n    assert captured.value.error.status_code == 401\n    assert captured.value.error.retryable is False\n    assert \'SECRET\' not in str(captured.value)\n\n\ndef test_adapter_surfaces_bounded_retry_hint_without_retrying_itself():\n    with fake_openai_server() as (base_url, requests):\n        adapter = OpenAICompatibleAdapter(\n            OpenAICompatibleConfig(base_url=base_url, api_key=\'explicit-key\')\n        )\n        with pytest.raises(ModelAdapterError) as captured:\n            asyncio.run(\n                complete(\n                    adapter,\n                    [AgentMessage.text(Role.USER, \'retry in runtime\')],\n                    ModelSpec(\'retry-me\'),\n                )\n            )\n\n    assert captured.value.error.code is ModelErrorCode.RATE_LIMIT\n    assert captured.value.error.retry_after_seconds == 7.0\n    assert len(requests) == 1\n\n\ndef test_adapter_normalizes_context_capacity_errors_for_compaction_recovery():\n    with fake_openai_server() as (base_url, requests):\n        adapter = OpenAICompatibleAdapter(\n            OpenAICompatibleConfig(base_url=base_url, api_key=\'explicit-key\')\n        )\n        with pytest.raises(ModelAdapterError) as captured:\n            asyncio.run(\n                complete(\n                    adapter,\n                    [AgentMessage.text(Role.USER, \'overflow safely\')],\n                    ModelSpec(\'overflow-me\'),\n                )\n            )\n\n    assert captured.value.error.code is ModelErrorCode.CONTEXT_OVERFLOW\n    assert captured.value.error.retryable is False\n    assert captured.value.error.status_code == 400\n    assert len(requests) == 1\n'


In [ ]:
CHAPTER_CONTRACT_TEST_SOURCE = 'from __future__ import annotations\n\nimport asyncio\n\nimport pytest\n\nfrom agent_harness import (\n    AgentMessage,\n    AgentRuntime,\n    AgentSession,\n    CompactionPolicy,\n    ModelEnd,\n    ModelRequest,\n    ModelSpec,\n    Role,\n    SessionBusyError,\n    StopReason,\n    TextDelta,\n)\n\n\ndef test_compaction_accepts_work_only_at_an_idle_settled_boundary() -> None:\n    class BlockingAdapter:\n        def __init__(self) -> None:\n            self.started = asyncio.Event()\n            self.release = asyncio.Event()\n\n        async def stream(self, request: ModelRequest):\n            self.started.set()\n            await self.release.wait()\n            yield TextDelta("settled")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = BlockingAdapter()\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/settled-boundary"))\n        )\n        handle = session.start("active")\n        await adapter.started.wait()\n\n        with pytest.raises(SessionBusyError, match="Settled Boundary"):\n            await session.compact()\n\n        adapter.release.set()\n        await handle.result()\n\n    asyncio.run(scenario())\n\n\ndef test_compaction_configuration_fails_before_model_work_is_accepted() -> None:\n    runtime = AgentRuntime(\n        type("UnusedAdapter", (), {"stream": lambda self, request: None})(),\n        ModelSpec("scripted/configuration"),\n        history=(\n            AgentMessage.text(Role.USER, "question"),\n            AgentMessage.text(Role.ASSISTANT, "answer"),\n        ),\n    )\n    session = AgentSession(runtime)\n\n    with pytest.raises(ValueError, match="focus cannot be empty"):\n        asyncio.run(session.compact("  "))\n    with pytest.raises(ValueError, match="keep_recent_tokens"):\n        CompactionPolicy(keep_recent_tokens=0)\n'


In [ ]:
PYPROJECT_SOURCE = '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "agent-harness"\nversion = "0.6.0"\ndescription = "Chapter 6 settled Compaction and context overflow recovery"\nreadme = "README.md"\nrequires-python = ">=3.11"\ndependencies = ["jsonschema>=4.23,<5", "openai>=1.40,<3"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.setuptools.package-data]\nagent_harness = ["py.typed"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n'


In [ ]:
README_SOURCE = "# Agent Harness — Chapter 6 Checkpoint\n\nThis cumulative Checkpoint adds Compaction as an `AgentSession` operation available only at a Settled Boundary. Manual Compaction accepts focus instructions; threshold Compaction uses adaptive reserve and retained-token budgets; normalized context overflow gets one separate recovery attempt rather than entering ordinary transient retry.\n\n`CompactionStrategy.plan(...)` selects a materialized Retained Tail without splitting an assistant Tool Call from its complete `ToolResultMessage` batch. `CompactionPolicy.resolve(...)` applies the small- and broad-context formulas from ADR 0048. Token estimation remains an explicit replaceable seam and carries no claim of Provider accounting.\n\nEach durable `CompactionCheckpoint` stores a versioned structured summary, estimated tokens before Compaction, Provider summary usage when available, its trigger, and the complete retained messages. Session entries remain append-only: `history()` navigates the original conversation, while `effective_history()` materializes the latest summary, Retained Tail, and later settlements for the next model request.\n\nSummary generation uses the Runtime's configured `ModelAdapter`, `ModelSpec`, `RetryPolicy`, sleeper, and cancellation path with `ModelOperation.COMPACTION`. Threshold failure records a warning and preserves history; failed overflow recovery terminates clearly with `compaction_failed`. An unknown context window disables threshold triggering with one warning while leaving overflow recovery available.\n\nBoth `MemorySessionStore` and `JSONLSessionStore` persist the same checkpoint contract. JSONL schema minor 1 adds Compaction records while continuing to read prior major-1 Session files.\n"


In [ ]:
from course.tools.checkpoint import checkpoint_drift, export_checkpoint

checkpoint_result = export_checkpoint(
    ROOT / 'course' / 'notebooks' / '06_context_compaction.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch06',
)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
assert checkpoint_drift(
    ROOT / 'course' / 'notebooks' / '06_context_compaction.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch06',
) == ()
checkpoint_result


## Public API Summary

Call `await session.compact(focus)` for manual Compaction. Configure `CompactionPolicy`, inject a `CompactionStrategy` or `TokenEstimator` when needed, and inspect `CompactionResult.checkpoint` plus `session.warnings`. Session stores expose `compactions()` and `effective_history()` while `history()` and historical leaf navigation continue to return the original settled conversation.

`ModelOperation.COMPACTION`, `CompactionTrigger`, and Compaction Runtime Events make summary work distinguishable for later tracing without turning diagnostics into recovery state.